In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import datetime
from tkcalendar import Calendar

class EarthquakeApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Otomatisasi Informasi Gempabumi Mingguan")
        self.root.geometry("800x600")
        self.root.resizable(False, False)
        
        self.start_date_var = tk.StringVar(value=datetime.now().strftime("%d/%m/%Y"))
        self.end_date_var = tk.StringVar(value=datetime.now().strftime("%d/%m/%Y"))

        self.style = ttk.Style()
        self.style.configure('TFrame', background='#f0f0f0')
        self.style.configure('TLabel', background='#f0f0f0', font=('Arial', 10))
        self.style.configure('Title.TLabel', font=('Arial', 14, 'bold'))
        self.style.configure('TButton', font=('Arial', 10), width=20, padding=5)
        self.style.configure('Print.TButton', background='#e1f5fe', font=('Arial', 10))

        self.main_container = ttk.Frame(root)
        self.main_container.pack(fill=tk.BOTH, expand=True)

        self.left_panel = ttk.Frame(self.main_container, width=200, padding=10)
        self.left_panel.pack(side=tk.LEFT, fill=tk.Y)

        self.right_panel = ttk.Frame(self.main_container, padding=10)
        self.right_panel.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

        title_label = ttk.Label(self.right_panel, text="Otomatisasi Informasi Gempabumi Mingguan", style='Title.TLabel')
        title_label.pack(pady=(0, 10))

        instruction_label = ttk.Label(self.right_panel, text="Petunjuk:", font=('Arial', 10, 'bold'))
        instruction_label.pack(anchor='w')

        self.instruction_text = tk.Text(self.right_panel, height=10, wrap=tk.WORD, font=('Arial', 10))
        self.instruction_text.pack(fill=tk.BOTH, expand=True)
        self.instruction_text.insert(tk.END, 
            "Silakan ikuti langkah-langkah berikut untuk menggunakan aplikasi ini:\n"
            "1. Siapkan Data Gempabumi yang sudah didownload dari SPK sebelumnya.\n"
            "2. Klik 'Pilih Tanggal' dan tentukan tanggal awal dan akhir periode analisis.\n"
            "3. Klik tombol 'Proses' untuk memulai analisis data gempabumi.\n"
            "4. Gunakan tombol cetak untuk mencetak hasil analisis, peta, laporan, press release, atau deskripsi.\n"
            "5. Klik 'Keluar' untuk menutup aplikasi."
        )

        self.create_buttons()

    def create_buttons(self):
        ttk.Button(self.left_panel, text="Pilih File", command=self.select_file).pack(pady=5)
        
        # Gabungan tombol pilih tanggal
        ttk.Button(self.left_panel, text="Pilih Tanggal", command=self.show_date_range_selector).pack(pady=5)
        ttk.Label(self.left_panel, textvariable=self.start_date_var).pack()
        ttk.Label(self.left_panel, textvariable=self.end_date_var).pack()

        ttk.Button(self.left_panel, text="Proses", command=self.process_data).pack(pady=20)

        print_buttons = [
            ("Cetak Analisis xlsx", self.print_analysis),
            ("Cetak Peta", self.print_map),
            ("Cetak Laporan", self.print_report),
            ("Cetak Press Release", self.print_press_release),
            ("Cetak Deskripsi", self.print_description)
        ]
        for text, command in print_buttons:
            ttk.Button(self.left_panel, text=text, command=command, style='Print.TButton').pack(pady=3)

        ttk.Button(self.left_panel, text="Keluar", command=self.root.destroy).pack(side=tk.BOTTOM, pady=20)

    def show_date_range_selector(self):
        top = tk.Toplevel(self.root)
        top.title("Pilih Tanggal Awal dan Akhir")

        ttk.Label(top, text="Tanggal Awal").grid(row=0, column=0, padx=10, pady=5)
        cal_start = Calendar(top, selectmode='day', date_pattern='dd/mm/yyyy')
        cal_start.grid(row=1, column=0, padx=10, pady=10)

        ttk.Label(top, text="Tanggal Akhir").grid(row=0, column=1, padx=10, pady=5)
        cal_end = Calendar(top, selectmode='day', date_pattern='dd/mm/yyyy')
        cal_end.grid(row=1, column=1, padx=10, pady=10)

        def set_dates():
            self.start_date_var.set(cal_start.get_date())
            self.end_date_var.set(cal_end.get_date())
            top.destroy()

        ttk.Button(top, text="Pilih", command=set_dates).grid(row=2, column=0, columnspan=2, pady=10)

    def select_file(self):
        global selected_files
        selected_files = filedialog.askopenfilenames(title="Pilih File", filetypes=(("Excel Files", "*.xlsx"), ("Semua File", "*.*")))
        if selected_files:
            messagebox.showinfo("File Terpilih", f"Jumlah file terpilih: {len(selected_files)}")

    def standardize_months(text):
        month_map = {
            'Jan': '01', 'January': '01', 'Januari': '01',
            'Feb': '02', 'February': '02', 'Februari': '02',
            'Mar': '03', 'March': '03', 'Maret': '03',
            'Apr': '04', 'April': '04',
            'May': '05', 'Mei': '05',
            'Jun': '06', 'June': '06', 'Juni': '06',
            'Jul': '07', 'July': '07', 'Juli': '07',
            'Aug': '08', 'August': '08', 'Agustus': '08',
            'Sep': '09', 'September': '09',
            'Oct': '10', 'October': '10', 'Oktober': '10',
            'Nov': '11', 'November': '11',
            'Dec': '12', 'December': '12', 'Desember': '12'
        }
        
        pattern = r'(\d{1,2})-([a-zA-Z]+)-(\d{2})'
        
        def replace_month(match):
            day = match.group(1)
            month = match.group(2)
            year = match.group(3)
            month_num = month_map.get(month.title(), month)  # Default to original if not found
            return f"{day}-{month_num}-{year}"
        
        return re.sub(pattern, replace_month, text)

    def process_data(self):
        global selected_files, tanggal_awal, tanggal_akhir, nama_folder
        locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

        start_date_str = start_date_entry.get()
        end_date_str = end_date_entry.get()

        tanggal_awal = pd.to_datetime(start_date_str, format='%d/%m/%Y')
        tanggal_akhir = pd.to_datetime(end_date_str, format='%d/%m/%Y')

        if not selected_files:
            messagebox.showerror("Error", "Pilih file terlebih dahulu!")
            return None, None

        column_name = 'Info Gempa' 

        def parse_text(text):
            # Standardize month names first
            standardized_text = standardize_months(text)
            
            pattern = r"Mag:(?P<magnitude>\d+\.\d+) , (?P<date>\d{2}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) WIB, Lok:(?P<latitude>\d+\.\d+) LS,(?P<longitude>\d+\.\d+) BT \((?P<description>.+?)\), Kedlmn:(?P<depth>\d+) Km"
            match = re.search(pattern, standardized_text)
            if match:
                result = match.groupdict()
                result['latitude'] = '-' + result['latitude']  # Tambahkan tanda negatif untuk LS
                return result
            return None

        combined_data = pd.DataFrame()
        
        for file in selected_files:
            df = pd.read_excel(file, header=1)
            # Apply standardization before parsing
            df[column_name] = df[column_name].astype(str).apply(standardize_months)
            parsed_data = df[column_name].dropna().apply(parse_text).apply(pd.Series)
            parsed_data = parsed_data.rename(columns={
                'date': 'Tanggal',
                'time': 'Waktu (WIB)',
                'latitude': 'Lintang',
                'longitude': 'Bujur',
                'magnitude': 'Magnitude',
                'depth': 'Kedalaman',
                'description': 'Keterangan'
            })

            parsed_data['Dirasakan'] = ''
            parsed_data['Tanggal'] = pd.to_datetime(parsed_data['Tanggal'], format='%d-%m-%y', errors='coerce')

            start_longitude = 113.21
            end_longitude = 117.31

            mask = (
                (parsed_data['Tanggal'] >= tanggal_awal) & 
                (parsed_data['Tanggal'] <= tanggal_akhir) &
                (parsed_data['Bujur'].astype(float) >= start_longitude) & 
                (parsed_data['Bujur'].astype(float) <= end_longitude)
            )
            filtered_data = parsed_data[mask]
            combined_data = pd.concat([combined_data, filtered_data])

        combined_data.to_csv('dataspk.csv', index=False)
        hasil_data = pd.read_csv('dataspk.csv')

        existing_file = 'dirasakan.xlsx'
        if not os.path.exists(existing_file):
            messagebox.showerror("Error", f"File '{existing_file}' tidak ditemukan!")
            return

        existing_data = pd.read_excel(existing_file)

        # Konversi tipe data kolom kunci agar konsisten
        key_columns = ['Tanggal', 'Waktu (WIB)', 'Magnitude', 'Lintang', 'Bujur', 'Kedalaman']
        for col in key_columns:
            if col in existing_data.columns:
                if col == 'Tanggal':
                    existing_data[col] = pd.to_datetime(existing_data[col], errors='coerce')
                else:
                    existing_data[col] = existing_data[col].astype(str)

            if col in hasil_data.columns:
                if col == 'Tanggal':
                    hasil_data[col] = pd.to_datetime(hasil_data[col], errors='coerce')
                else:
                    hasil_data[col] = hasil_data[col].astype(str)

        merged_data = pd.merge(existing_data, hasil_data, on=key_columns, how='outer', suffixes=('', '_new'))

        for col in hasil_data.columns:
            if col not in key_columns:
                merged_data[col] = merged_data[col + '_new'].combine_first(merged_data[col])
                merged_data.drop(columns=[col + '_new'], inplace=True)

        merged_data['Datetime'] = pd.to_datetime(merged_data['Tanggal'].astype(str) + ' ' + merged_data['Waktu (WIB)'], errors='coerce')
        merged_data = merged_data.sort_values(by='Datetime').drop(columns=['Datetime'])
        merged_data['Tanggal'] = pd.to_datetime(merged_data['Tanggal']).dt.date

        column_order = ['No', 'Tanggal', 'Waktu (WIB)', 'Lintang', 'Bujur', 'Kedalaman', 'Magnitude', 'Keterangan', 'Dirasakan']
        merged_data.insert(0, 'No', range(1, len(merged_data) + 1))
        merged_data = merged_data[column_order]
        
        merged_data.to_csv('datalengkap.csv', index=False)

        if tanggal_awal.month == tanggal_akhir.month:
            nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
        else:
            nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

        if not os.path.exists(nama_folder):
            os.makedirs(nama_folder)

        output_file = os.path.join(nama_folder, f'Data Gempabumi Periode {nama_folder}.xlsx')
        merged_data.to_excel(output_file, index=False)

        messagebox.showinfo("Proses", f"Data berhasil diproses dan disimpan ke:\n1. {output_file}\n2. datalengkap.csv di direktori script.")

    def print_analysis(self):
        def read_csv_and_analyze(tanggal_awal, tanggal_akhir):
        
            # Step 1: Read CSV file
            file_path = "datalengkap.csv"
            data = pd.read_csv(file_path)
            
            # Konversi kolom Tanggal ke datetime dan ambil hanya tanggalnya
            data['Tanggal'] = pd.to_datetime(data['Tanggal']).dt.date
            tanggal_awal = pd.to_datetime(tanggal_awal).date()
            tanggal_akhir = pd.to_datetime(tanggal_akhir).date()
            
            # Filter data berdasarkan tanggal yang dipilih
            data = data[(data['Tanggal'] >= tanggal_awal) & (data['Tanggal'] <= tanggal_akhir)]
            all_dates = pd.date_range(start=tanggal_awal, end=tanggal_akhir, freq='D')
            periode = all_dates.strftime('%Y-%m-%d')  # Format tanggal sebagai string

            # Drop the 'Magnitude Category' and 'Depth Category' columns
            output_data = data.drop(columns=['Magnitude Category', 'Depth Category'], errors='ignore')

            # Create magnitude and depth categories
            categories_magnitude = ['M<3', '3≤M<5', 'M≥5']
            data['Magnitude'] = pd.to_numeric(data['Magnitude'], errors='coerce')
            data['Magnitude Category'] = pd.cut(data['Magnitude'], bins=[0, 3, 5, float('inf')], labels=categories_magnitude)

            categories_depth = ['D≤60 km', '60<D≤300 km', 'D>300 km']
            data['Kedalaman'] = pd.to_numeric(data['Kedalaman'], errors='coerce')
            data['Depth Category'] = pd.cut(data['Kedalaman'], bins=[0, 60, 300, float('inf')], labels=categories_depth)      
            
            # Group data by day and categories for analysis
            # magnitude_summary = (data.groupby([data['Tanggal'], 'Magnitude Category']).size().unstack(fill_value=0).reindex(index=all_dates, fill_value=0).reindex(columns=categories_magnitude, fill_value=0))
            # depth_summary = (data.groupby([data['Tanggal'], 'Depth Category']).size().unstack(fill_value=0).reindex(index=all_dates, fill_value=0).reindex(columns=categories_depth, fill_value=0))
            
            magnitude_summary = (data.groupby([data['Tanggal'], 'Magnitude Category'])
                                .size()
                                .unstack(fill_value=0)
                                .reindex(index=(pd.date_range(start=tanggal_awal, end=tanggal_akhir, freq='D')), fill_value=0)
                                .reindex(columns=categories_magnitude, fill_value=0))

            depth_summary = (data.groupby([data['Tanggal'], 'Depth Category'])
                            .size()
                            .unstack(fill_value=0)
                            .reindex(index=(pd.date_range(start=tanggal_awal, end=tanggal_akhir, freq='D')), fill_value=0)
                            .reindex(columns=categories_depth, fill_value=0))

            # Add totals row and columns for 'Gempa Dirasakan' and 'Gempa Merusak'
            magnitude_summary['Jumlah Total'] = magnitude_summary.sum(axis=1)
            magnitude_summary['Gempa Dirasakan'] = 0
            magnitude_summary['Gempa Merusak'] = 0
            magnitude_summary.loc['Jumlah Gempa'] = magnitude_summary.sum()

            depth_summary['Jumlah Total'] = depth_summary.sum(axis=1)
            depth_summary['Gempa Dirasakan'] = 0
            depth_summary['Gempa Merusak'] = 0
            depth_summary.loc['Jumlah Gempa'] = depth_summary.sum()

            # Mengisi kolom 'Gempa Dirasakan' jika kolom 'Dirasakan' terisi
            for index, row in data.iterrows():
                if pd.notna(row['Dirasakan']) and row['Dirasakan'].strip() != '':
                    if row['Tanggal'] in magnitude_summary.index:
                        magnitude_summary.at[row['Tanggal'], 'Gempa Dirasakan'] += 1
                    if row['Tanggal'] in depth_summary.index:
                        depth_summary.at[row['Tanggal'], 'Gempa Dirasakan'] += 1

            magnitude_summary.at['Jumlah Gempa', 'Gempa Dirasakan'] = magnitude_summary['Gempa Dirasakan'].sum()
            depth_summary.at['Jumlah Gempa', 'Gempa Dirasakan'] = depth_summary['Gempa Dirasakan'].sum()

            if tanggal_awal.month == tanggal_akhir.month:
                nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
            else:
                nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

            # Untuk menangani row 'Jumlah Gempa' yang bukan tanggal
            magnitude_summary.index = [x.strftime('%Y-%m-%d') if isinstance(x, pd.Timestamp) else x for x in magnitude_summary.index]
            depth_summary.index = [x.strftime('%Y-%m-%d') if isinstance(x, pd.Timestamp) else x for x in depth_summary.index]

            # Save the analysis to Excel
            output_file = os.path.join(nama_folder, f'Analisis Gempabumi Periode {nama_folder}.xlsx')
            with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
                output_data.to_excel(writer, sheet_name='Earthquake Data', index=False)
                magnitude_summary.to_excel(writer, sheet_name='Magnitude Summary')
                depth_summary.to_excel(writer, sheet_name='Depth Summary')
                plot_charts(magnitude_summary, depth_summary, writer, nama_folder, tanggal_awal, tanggal_akhir)

            # Autofit columns
            autofit_columns(output_file, 'Earthquake Data')
            autofit_columns(output_file, 'Magnitude Summary')
            autofit_columns(output_file, 'Depth Summary')

            messagebox.showinfo("Info", f"Analisis berhasil disimpan di:\n{output_file}")

        def plot_charts(magnitude_summary, depth_summary, writer, nama_folder, tanggal_awal, tanggal_akhir,):
            # Set the size for both bar and pie charts
            bar_chart_size = (16, 10)
            pie_chart_size = (5, 5)

            # ================== MAGNITUDE BAR CHART ==================
            categories = ['M<3', '3≤M<5', 'M≥5']
            colors = ['green', 'yellow', 'red']
            
            # Remove total row
            magnitude_summary_without_total = magnitude_summary.drop(index='Jumlah Gempa', errors='ignore')
            
            # Reindex with all dates and fill missing with 0
            magnitude_summary_without_total.index = pd.to_datetime(magnitude_summary_without_total.index)
            magnitude_summary_without_total.index = magnitude_summary_without_total.index.strftime('%d-%m-%Y')

            # Plot setup
            fig, ax = plt.subplots(figsize=(12, 6))
            bar_width = 0.25
            x = np.arange(len(magnitude_summary_without_total.index))

            # Plot each category
            for i, category in enumerate(categories):
                ax.bar(x + i * bar_width, 
                    magnitude_summary_without_total[category], 
                    bar_width, 
                    label=category, 
                    color=colors[i], 
                    edgecolor='black')

            # Chart formatting
            ax.set_title('Frekuensi Gempa Berdasarkan Magnitudo',fontsize=14, pad=20)
            ax.set_xlabel('Tanggal', fontsize=12)
            ax.set_ylabel('Jumlah Kejadian', fontsize=12)
            
            # Set x-ticks to show all dates
            ax.set_xticks(x + bar_width)
            
            # Update x-tick labels dengan format baru
            ax.set_xticklabels(magnitude_summary_without_total.index, rotation=45, ha='right', fontsize=8)
            
            ax.legend(title='Kategori Magnitudo', fontsize=10)
            ax.grid(axis='y', linestyle='--', alpha=0.7)

            # Add value labels
            for p in ax.patches:
                height = p.get_height()
                if height > 0:  # Hanya tampilkan label jika nilai > 0
                    ax.annotate(f'{int(height)}', 
                            (p.get_x() + p.get_width() / 2, height),
                            ha='center', va='bottom', fontsize=8)

            plt.tight_layout()
            plt.savefig(os.path.join(nama_folder, 'diagbat_mag.png'), dpi=300)
            plt.close()

            # Insert to Excel
            worksheet_mag = writer.sheets['Magnitude Summary']
            worksheet_mag.insert_image('A10', os.path.join(nama_folder, 'diagbat_mag.png'), 
                                    {'x_scale': 1.2, 'y_scale': 1.4})

            # Buat diagram lingkaran magnitude
            magnitude_total = magnitude_summary.loc['Jumlah Gempa', ['M<3', '3≤M<5', 'M≥5']]
            labels = ['M<3', '3≤M<5', 'M≥5']
            colors = ['green', 'yellow', 'red']

            # Membuat figure dan pie chart
            fig, ax = plt.subplots(figsize=(6, 6))

            # Fungsi untuk menampilkan persentase dan jumlah kejadian di setiap segmen
            def func(pct, allvals):
                absolute = int(np.round(pct/100.*np.sum(allvals)))
                if absolute > 0:
                    return f'{pct:.1f}%\n({absolute} kejadian)'
                else:
                    return ''  # Jika tidak ada kejadian, label tidak ditampilkan

            # Membuat pie chart
            wedges, texts, autotexts = ax.pie(
                magnitude_total,
                labels=labels,
                colors=colors,
                startangle=85,
                autopct=lambda pct: func(pct, magnitude_total),  # Menampilkan persentase dan jumlah kejadian
                pctdistance=0.875,  # Jarak persentase dari pusat pie
                labeldistance=9  # Jarak label dari pusat pie (aslinya 1.17)
            )

            # Mengatur ukuran font dan latar belakang putih untuk label dan persentase
            for text in texts:
                text.set_fontsize(12)
                text.set_bbox(dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.2'))

            for autotext in autotexts:
                autotext.set_fontsize(12)
                autotext.set_bbox(dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.2'))

            plt.title('Distribusi Magnitudo', fontsize=11)
            ax.legend(title='Kategori Magnitudo', fontsize=7, title_fontsize='9', loc='lower right')
            plt.tight_layout()

            plt.savefig(os.path.join(nama_folder, 'diaglingk_mag.png'))

            # Insert the magnitude pie chart into Excel
            worksheet_mag.insert_image('W10', os.path.join(nama_folder, 'diaglingk_mag.png'), {'x_scale': 1.2, 'y_scale': 1.2})
            
            # ================== DEPTH BAR CHART ==================
            categories_depth = ['D≤60 km', '60<D≤300 km', 'D>300 km']
            colors_depth = ['red', 'yellow', 'green']
            
        # Remove total row
            depth_summary_without_total = depth_summary.drop(index='Jumlah Gempa', errors='ignore')
            
            # Reindex with all dates and fill missing with 0
            depth_summary_without_total.index = pd.to_datetime(depth_summary_without_total.index)
            depth_summary_without_total.index = depth_summary_without_total.index.strftime('%d-%m-%Y')

            # Plot setup
            fig, ax = plt.subplots(figsize=(12, 6))
            bar_width = 0.25
            x = np.arange(len(depth_summary_without_total.index))
            
            # Plot each category
            for i, category in enumerate(categories_depth):
                ax.bar(x + i * bar_width, 
                    depth_summary_without_total[category], 
                    bar_width, 
                    label=category, 
                    color=colors_depth[i], 
                    edgecolor='black')

            # Chart formatting
            ax.set_title('Frekuensi Gempa Berdasarkan Kedalaman' ,fontsize=14, pad=20)
            ax.set_xlabel('Tanggal', fontsize=12)
            ax.set_ylabel('Jumlah Kejadian', fontsize=12)
            
            # Set x-ticks to show all dates
            ax.set_xticks(x + bar_width)
            
            # Update x-tick labels dengan format baru
            ax.set_xticklabels(depth_summary_without_total.index, rotation=45, ha='right', fontsize=8)
            
            ax.legend(title='Kategori Kedalaman', fontsize=10)
            ax.grid(axis='y', linestyle='--', alpha=0.7)

            for p in ax.patches:
                height = p.get_height()
                if height > 0:  # Hanya tampilkan label jika nilai > 0
                    ax.annotate(f'{int(height)}', 
                            (p.get_x() + p.get_width() / 2, height),
                            ha='center', va='bottom', fontsize=8)

            plt.tight_layout()
            plt.savefig(os.path.join(nama_folder, 'diagbat_depth.png'), dpi=300)
            plt.close()

            worksheet_depth = writer.sheets['Depth Summary']
            worksheet_depth.insert_image('A10', os.path.join(nama_folder, 'diagbat_depth.png'), 
                                    {'x_scale': 1.2, 'y_scale': 1.4})

            # buat digram lingkaran depth
            depth_total = depth_summary.loc['Jumlah Gempa', ['D≤60 km', '60<D≤300 km', 'D>300 km']]
            labels = ['D≤60 km', '60<D≤300 km', 'D>300 km']
            colors = ['red', 'yellow', 'green']
            
            # Membuat figure dan pie chart
            fig, ax = plt.subplots(figsize=(6, 6))
            
            # Fungsi untuk menampilkan persentase dan jumlah kejadian di setiap segmen
            def func(pct, allvals):
                absolute = int(np.round(pct/100.*np.sum(allvals)))
                if absolute > 0:
                    return f'{pct:.1f}%\n({absolute} kejadian)'
                else:
                    return ''  # Jika tidak ada kejadian, label tidak ditampilkan
                    
            # Membuat pie chart
            wedges, texts, autotexts = ax.pie(
                depth_total,
                labels=labels,
                colors=colors,
                startangle=85,
                autopct=lambda pct: func(pct, magnitude_total),  # Menampilkan persentase dan jumlah kejadian
                pctdistance=0.975,  # Jarak persentase dari pusat pie
                labeldistance=9  # Jarak label dari pusat pie (aslinya 1.15)
            )

            # Mengatur ukuran font dan latar belakang putih untuk label dan persentase
            for text in texts:
                text.set_fontsize(12)
                text.set_bbox(dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.2'))

            for autotext in autotexts:
                autotext.set_fontsize(12)
                autotext.set_bbox(dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.2'))

            plt.title('Distribusi Kedalaman', fontsize=11)
            ax.legend(title='Kategori Kedalaman', fontsize=7, title_fontsize='9', loc='lower right')
            plt.tight_layout()
            plt.savefig(os.path.join(nama_folder, 'diaglingk_depth.png'))

            # Insert the magnitude pie chart into Excel
            worksheet_depth.insert_image('W10', os.path.join(nama_folder, 'diaglingk_depth.png'), {'x_scale': 1.2, 'y_scale': 1.2})   
        
        def autofit_columns(file_path, sheet_name):
            """
            Menyesuaikan lebar kolom di Excel berdasarkan panjang konten.
            """
            workbook = load_workbook(file_path)  # Membuka workbook yang telah disimpan
            worksheet = workbook[sheet_name]  # Mendapatkan worksheet berdasarkan nama sheet

            for col in worksheet.columns:
                max_length = 0
                column = col[0].column_letter  # Mendapatkan huruf kolom, misalnya 'A', 'B', dst.
                for cell in col:
                    try:
                        # Mencari panjang maksimum dari setiap kolom
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = (max_length + 2)  # Menambah padding
                worksheet.column_dimensions[column].width = adjusted_width  # Mengatur lebar kolom

            workbook.save(file_path)  # Menyimpan workbook setelah penyesuaian
            workbook.close()  # Menutup workbook untuk memastikan semua perubahan disimpan
        
        # Panggil fungsi utama dengan tanggal yang dipilih
        read_csv_and_analyze(tanggal_awal, tanggal_akhir)

    def print_map(self):
        fault="D:/Seismioto/Bahan/fault.gmt"
        trench="D:/Seismioto/Bahan/trench.gmt"
        legenda="D:/Seismioto/Bahan/legenda.gmt"
        #membuat file .dat
        df = pd.read_csv('datalengkap.csv')
        
        # Memilih kolom yang diinginkan
        selected_columns = df[['Bujur','Lintang','Kedalaman', 'Magnitude']]

        # Menulis data ke file DAT
        with open('inputpeta.dat', 'w') as file:
            for index, row in selected_columns.iterrows():
                line = '  '.join(row.astype(str))  # Menggabungkan semua elemen baris menjadi satu string dengan spasi sebagai pemisah
                file.write(line + '\n')
                
        # Membaca data dari hasil2.csv
        data = pd.read_csv('datalengkap.csv')

        # Buat figure
        fig = pygmt.Figure()

        # Buat color palette untuk kedalaman gempa
        pygmt.makecpt(cmap="geo", series=[-7000, 4000])
        pygmt.makecpt(cmap="red,yellow,green", series="0,60,300,1000", output="quakes.cpt")
        # start_date_formatted = start_date_entry.get().replace('/', '-')  # Ganti format tanggal
        # end_date_formatted = end_date_entry.get().replace('/', '-')
        #title_text = f"Peta Seismisitas Wilayah Bali dan Sekitarnya \nPeriode {start_date_formatted} hingga {end_date_formatted}"
        data['Magnitude'] = pd.to_numeric(data['Magnitude'], errors='coerce')  # Ganti nilai tidak valid dengan NaN

        # Plot peta dasar dengan topografi
        fig.grdimage(
            grid= file_grid,
            region=[113.21, 117.31, -12, -7],
            projection="M17c",
            shading=True,
            frame=["xa2g2", "ya2g2"])#'+t"Peta Seismisitas"'])
        
        # Plot fault lines
        
        fig.plot(data=fault, pen="0.6,black")
        fig.plot(data=trench, pen="4,black")

        fig.plot(x=[114.8, 115.4], y=[-11.5, -7.5], projection="M", pen=2)
        fig.text(x=114.9, y=-11.6, text="A", font="18,Helvetica")
        fig.text(x=115.55, y=-7.5, text="A1", font="18,Helvetica")

        # Plot gempa yang tidak dirasakan (kolom 'Dirasakan' kosong)
        gempa_tidak_dirasakan = data[data['Dirasakan'].isna()]
        if not gempa_tidak_dirasakan.empty:
            fig.plot(
                x=gempa_tidak_dirasakan['Bujur'],
                y=gempa_tidak_dirasakan['Lintang'],
                size=0.08*gempa_tidak_dirasakan['Magnitude'],
                style="cc",  # Lingkaran
                fill=gempa_tidak_dirasakan['Kedalaman'].astype(float),  # Warna berdasarkan kedalaman
                cmap="quakes.cpt",  # Gunakan colormap yang telah ditentukan
                pen="black"
            )
        else:
            print("ada gempa bumi yang dirasakan untuk diplot.")

        # Plot gempa yang dirasakan (kolom 'Dirasakan' terisi)
        gempa_dirasakan = data[~data['Dirasakan'].isna()]
        if not gempa_dirasakan.empty:
            fig.plot(
                x=gempa_dirasakan['Bujur'],
                y=gempa_dirasakan['Lintang'],
                size=0.08*gempa_dirasakan['Magnitude'],
                style="a0.5c",  # Simbol bintang
                fill="red",  # Warna bintang merah
                pen="1p,black"  # Garis tepi hitam dengan ketebalan 1 point
            )
        else:
            print("tidak ada gempa bumi yang dirasakan untuk diplot.")
        
        # Menambahkan colorbar
        fig.image(logobmkg, position='n1.1/0.85+w1.1i')
        fig.image(matangin, position='n1.04/0.7+w1.1i')
        fig.basemap(map_scale="n1.12/0.65+w80k+f0.1p+lKm")
        fig.colorbar(
            frame=["x+lKetinggian", "y"],  # Label untuk sumbu x dan y
            position="n1.24/0.63+w4c/0.5c")  # Posisi colorbar (JMR = tengah-kanan), lebar 0.5 cm, panjang 10 cm
        fig.legend(spec=legenda, position="n1.03/0.04+w5c",box="+gwhite+p1p,black")
        with fig.inset(position="n1.03/0.28+w5c/2.5c", box="+pblack"):
            fig.coast(
                region=[97, 140, -15, 7],
                shorelines='0.5p,black',
                projection="M5c",
                land="green",
                water="lightblue"
            )
            fig.plot(x=[117.31, 113.21, 113.21, 117.31, 117.31], y=[-7, -7, -12, -12, -7], pen="1p,red")

        # Membuat cross section
        pygmt.project(
            data="inputpeta.dat",
            unit=True,
            center=[114.8, -11.5],
            endpoint=[115.4, -7.5],
            width=[-200, 200],
            convention="pz",
            outfile="crsx.dat"
        )

        fig.shift_origin(yshift=9, xshift=17.5)
        fig.plot(
            x=[113, 113, 114.18, 114.18],  # Koordinat X dari sudut-sudut persegi
            y=[-11.4, -12, -12, -11.4],  # Koordinat Y dari sudut-sudut persegi
            fill="white",  # Warna latar belakang
            close=True  # Menutup jalur untuk membuat persegi berisi warna 
        )

        # Buat peta cross section dengan basemap
        fig.basemap(
            projection="X4c/-2.5c",
            region=[0, 450, 0, 350],
            frame=['xafg+lDistance (km)','yafg+lDepth (km)', "wsEN"]
        )

        # Membuat garis cross section   
        cross = pd.read_csv('crsx.dat', delimiter='\s+', header=None)
        # Define your column names
        header = ['Distance', 'Dept', 'Mag']
        
        # Assign the header
        cross.columns = header

        fig.plot(x=cross.Distance, 
                y=cross.Dept, 
                projection="X4/-2.5", 
                size =0.035 * (2 * cross.Mag), style="cc", 
                fill = cross.Dept, 
                cmap = "quakes.cpt", 
                pen = 'black'),
                #region = region_cs)
        fig.text(x=20, y=25, text="A", font="11,Helvetica")
        fig.text(x=370, y=25, text="A1", font="11,Helvetica")

        # Tentukan nama folder berdasarkan tanggal awal dan akhir
        if tanggal_awal.month == tanggal_akhir.month:
            nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
        else:
            nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

        output_map_filename = os.path.join(nama_folder, f'Peta Seismisitas Periode {nama_folder}.png')
        # Menyimpan gambar
        fig.savefig(fname=output_map_filename)
        #fig.show()
        
        messagebox.showinfo("Cetak Peta", "Peta Berhasil di Cetak")

    def print_report(self):
         # Set locale to Indonesian for date formatting
        locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

        # Fungsi untuk memilih file
        def select_file():
            root = tk.Tk()
            root.withdraw()  # Menyembunyikan jendela utama
            file_path = filedialog.askopenfilename(title="Pilih file laporan_gempa.xlsx", filetypes=[("Excel files", "*.xlsx")])
            return file_path

        # Memilih file Excel
        file_path = select_file()
        if not file_path:
            print("Tidak ada file yang dipilih.")
        else:
            # Load the Excel file
            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])
            
            # Tentukan nama folder berdasarkan tanggal awal dan akhir
            if tanggal_awal.month == tanggal_akhir.month:
                nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
            else:
                nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

            # Ambil tanggal awal dan akhir
            start_date = tanggal_awal
            end_date = tanggal_akhir
    
            # Path to images
            logo_path = "D:/Seismioto/Bahan/LogoBMKG.png"  # Ganti dengan path logo BMKG
            magnitude_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_mag.png"
            magnitude_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_mag.png"
            depth_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_depth.png"
            depth_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_depth.png"
            seismic_map_path = f"D:/Seismioto/Pengolahan/{nama_folder}/Peta Seismisitas Periode {nama_folder}.png"
        
            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

            # Calculate the required values
            total_earthquakes = len(earthquake_data)
            min_magnitude = earthquake_data['Magnitude'].min()
            max_magnitude = earthquake_data['Magnitude'].max()

            # Format tanggal dengan nama bulan dalam bahasa Indonesia
            start_date_formatted = start_date.strftime('%d %B %Y')  # contoh format: 01 Januari 2024
            end_date_formatted = end_date.strftime('%d %B %Y')      # contoh format: 05 September 2024
            #date_formatted=date1.strftime('%d %B %Y')
            #print(date_formatted)

            # Menentukan tanggal awal bulan untuk memulai perhitungan minggu
            start_of_month = start_date.replace(day=1)

            # Hitung nomor minggu relatif terhadap awal bulan
            week_number = (start_date - start_of_month).days // 7 + 1

            # Format bulan dalam bahasa Indonesia
            bulan_formatted = start_date.strftime('%B')

            # Menghitung jumlah kejadian gempabumi total
            total_earthquakes = len(earthquake_data)

            # Menghitung jumlah gempa dengan lat > -9 dan kedalaman < 60 km
            shallow_earthquake_count = earthquake_data[(earthquake_data['Lintang'] > -9) & 
                                                    (earthquake_data['Kedalaman'] < 60)].shape[0]

            # Menghitung jumlah gempa di selatan Pulau Jawa, Bali, dan Lombok
            southern_earthquake_count = total_earthquakes - shallow_earthquake_count

            # Magnitude categories
            magnitude_below_3 = earthquake_data[earthquake_data['Magnitude'] <= 3]
            magnitude_3_to_5 = earthquake_data[(earthquake_data['Magnitude'] > 3) & (earthquake_data['Magnitude'] <= 5)]
            magnitude_above_5 = earthquake_data[earthquake_data['Magnitude'] > 5]

            count_below_3 = len(magnitude_below_3)
            count_3_to_5 = len(magnitude_3_to_5)
            count_above_5 = len(magnitude_above_5)

            percentage_below_3 = int((count_below_3 / total_earthquakes) * 100)
            percentage_3_to_5 = int((count_3_to_5 / total_earthquakes) * 100)
            percentage_above_5 = int((count_above_5 / total_earthquakes) * 100)

            # Magnitude categories
            depth_below_60 = earthquake_data[earthquake_data['Kedalaman'] < 60]
            depth_60_to_300 = earthquake_data[(earthquake_data['Kedalaman'] >= 60) & (earthquake_data['Kedalaman'] <= 300)]
            depth_above_300 = earthquake_data[earthquake_data['Kedalaman'] > 300]

            count_depth_below_60 = len(depth_below_60)
            count_depth_60_to_300 = len(depth_60_to_300)
            count_depth_above_300 = len(depth_above_300)

            percentage_depth_below_60 = int((count_depth_below_60 / total_earthquakes) * 100)
            percentage_depth_60_to_300 = int((count_depth_60_to_300 / total_earthquakes) * 100)
            percentage_depth_above_300 = int((count_depth_above_300 / total_earthquakes) * 100)

            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Create a new Word document
            document = Document()

            # Ukuran kertas F4 (21.0 cm x 33.0 cm)
            f4_width = Cm(21.0)
            f4_height = Cm(33.0)

            # Mengatur ukuran kertas F4 untuk setiap section di dokumen
            for section in document.sections:
                # Mengatur orientasi potrait untuk F4
                section.orientation = WD_ORIENT.PORTRAIT  # Bisa diubah ke WD_ORIENT.LANDSCAPE jika ingin landscape
                section.page_width = f4_width
                section.page_height = f4_height

                # Mengatur margin halaman (optional, bisa disesuaikan)
                section.top_margin = Cm(2.5)     # Marginn atas
                section.bottom_margin = Cm(2.5)  # Marginn bawah
                section.left_margin = Cm(2.5)    # Marginn kiri
                section.right_margin = Cm(2.5)   # Marginn kanan

            # Add Logo at the top
            # logo_paragraph = document.add_paragraph()
            # logo_run = logo_paragraph.add_run()
            # logo_run.add_picture(logo_path, width=Inches(1))  # Menambahkan logo dengan lebar 1.5 inches
            # logo_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # # Add Title
            title = document.add_paragraph()
            title_run = title.add_run(f'STASIUN GEOFISIKA DENPASAR                                                                                              LAPORAN AKTIVITAS SEISMISITAS WILAYAH BALI DAN SEKITARNYA                                                                                              PERIODE {start_date_formatted} – {end_date_formatted}')
            title_run.bold = True
            title_run.font.size = Pt(14)
            title_run.font.all_caps = True  # Membuat semua huruf kapital
            title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add Subtitle
            # subtitle = document.add_paragraph()
            # subtitle_run = subtitle.add_run(f'STASIUN GEOFISIKA DENPASAR                                                                                              LAPORAN AKTIVITAS SEISMISITAS WILAYAH BALI DAN SEKITARNYA                                                                                              PERIODE {start_date_formatted} – {end_date_formatted}')
            # subtitle_run.font.size = Pt(12)
            # subtitle_run.font.all_caps = True  # Membuat semua huruf kapital
            # subtitle.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # # Add Date Period
            # period = document.add_paragraph()
            # period_run = period.add_run(f'PERIODE {start_date_formatted} – {end_date_formatted}')
            # period_run.font.size = Pt(12)
            # period_run.font.all_caps = True  # Membuat semua huruf kapital
            # period.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')

            # Teks Deskriptif Magnitudo dengan perhitungan minggu bulanan
            magnitude_descriptive_text = (
                f"Berdasarkan data Stasiun Geofisika Denpasar selama minggu ke-{week_number} bulan "
                f"{bulan_formatted} {start_date.year}, "
                f"di daerah Bali dan sekitarnya telah terjadi {total_earthquakes} kejadian gempabumi dengan magnitudo bervariasi "
                f"mulai dari M {min_magnitude} sampai M {max_magnitude}. Kejadian gempabumi dengan magnitudo M<3 sejumlah "
                f"{count_below_3} kejadian atau {percentage_below_3}% dari total kejadian gempabumi. Sedangkan untuk M 3 – 5 "
                f"sejumlah {count_3_to_5} kejadian atau {percentage_3_to_5}% dari total kejadian gempabumi "
            )

            # Conditional text for magnitude above 5
            if count_above_5 > 0:
                magnitude_descriptive_text += (
                    f"dan kejadian gempa M>5 sejumlah {count_above_5} kejadian atau {percentage_above_5}% dari total kejadian gempabumi."
                )
            else:
                magnitude_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan magnitudo M>5."

            # Menambahkan paragraf dengan teks deskriptif magnitudo
            magnitude_paragraph = document.add_paragraph(magnitude_descriptive_text)

            # Mengatur alignment paragraf menjadi justify
            magnitude_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Add Magnitude Bar Chart Image
            try:
                document.add_paragraph().add_run().add_picture(magnitude_bar_chart_path, width=Inches(6.56))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM BATANG MAGNITUDO TIDAK DITEMUKAN').bold = True

            # Add Magnitude Pie Chart Image
            try:
                document.add_paragraph().add_run().add_picture(magnitude_pie_chart_path, width=Inches(3.15))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM LINGKARAN MAGNITUDO TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Magnitudo
            document.add_paragraph('Gambar 1. Histogram gempabumi harian berdasarkan magnitudo (atas) dan diagram gempabumi berdasarkan magnitudo (bawah)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Add space
            document.add_paragraph('')

            # Teks Deskriptif Kedalaman
            depth_descriptive_text = (
                f"Berdasarkan kedalaman, gempabumi yang mendominasi adalah kejadian gempabumi {dominant_category}  dari total kejadian gempabumi. "
                f"Jumlah gempabumi dangkal sebanyak {count_depth_below_60} kejadian gempabumi, "
                f"terdapat sebanyak {count_depth_60_to_300} kejadian gempabumi dengan kedalaman menengah 60 km - 300 km atau {percentage_depth_60_to_300}% dari total kejadian gempabumi, "
            )

            # Conditional text for depth above 300 km
            if count_depth_above_300 > 0:
                depth_descriptive_text += (
                    f"dan kejadian gempa dalam sejumlah {count_depth_above_300} kejadian atau {percentage_depth_above_300}% dari total kejadian gempabumi."
                )
            else:
                depth_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan kedalaman dalam >300 km."

            depth_paragraph = document.add_paragraph(depth_descriptive_text)
            depth_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Add Depth Bar Chart Image
            try:
                document.add_paragraph().add_run().add_picture(depth_bar_chart_path, width=Inches(6.56))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM BATANG KEDALAMAN TIDAK DITEMUKAN').bold = True

            # Add Depth Pie Chart Image
            try:
                document.add_paragraph().add_run().add_picture(depth_pie_chart_path, width=Inches(3.15))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM LINGKARAN KEDALAMAN TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Kedalaman (akan diisi sesuai format)
            document.add_paragraph('Gambar 2.  Histogram gempabumi harian berdasarkan kedalaman (atas) dan diagram gempabumi berdasarkan kedalaman (bawah)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add Seismic Map Image
            try:
                document.add_paragraph().add_run().add_picture(seismic_map_path, width=Inches(6))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('PETA SEISMISITAS TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Peta Seismisitas
            peta_seismisitas_text = (
                f"Gambar 3. Peta seismisitas Bali dan sekitarnya periode {start_date_formatted} - {end_date_formatted}"
            )
            peta_seismisitas_paragraph = document.add_paragraph(peta_seismisitas_text)
            peta_seismisitas_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')

            # Teks analisis yang diminta
            analisis_text1 = (
                f"Selama satu minggu terakhir, gempabumi yang terjadi di Bali dan sekitarnya dikelompokan berdasarkan sumbernya (Gambar 3):\n"
            )    

            # Menambahkan paragraf analisis ke dalam dokumen
            analisis_paragraph1 = document.add_paragraph(analisis_text1)
            analisis_paragraph1.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

            analisis_text2 = (
                f"1. Sebanyak {southern_earthquake_count} kejadian gempabumi terjadi di selatan Pulau Jawa, Bali, dan Lombok "
                f"yang diperkirakan berasosiasi dengan subduksi Indo Australia-Eurasia.\n"
                f"2. Sebanyak {shallow_earthquake_count} kejadian gempabumi terjadi menyebar di wilayah Pulau Bali, Lombok dan sekitarnya "
                f"diperkirakan berkaitan dengan sesar di belakang busur Flores atau Flores Back Arc Thrust dan aktivitas sesar aktif."
            )

            # Menambahkan paragraf analisis ke dalam dokumen
            analisis_paragraph2 = document.add_paragraph(analisis_text2)
            analisis_paragraph2.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

            # Menghitung jumlah kejadian gempa yang dirasakan
            earthquakes_felt = earthquake_data[earthquake_data['Dirasakan'].notna()]
            total_felt = len(earthquakes_felt)

            # Menyusun narasi pembuka
            if total_felt > 0:
                felt_intro = (
                    f"Selama periode ini terdapat {total_felt} kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )
            else:
                felt_intro = (
                    f"Selama periode ini tidak terdapat kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )

            # Menambahkan narasi pembuka ke dokumen
            felt_intro_paragraph = document.add_paragraph(felt_intro)
            felt_intro_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Menyusun detail setiap kejadian gempa yang dirasakan dalam format list dengan nomor urut
            if total_felt > 0:
                # Inisialisasi counter untuk nomor urut
                counter = 1

                for index, row in earthquakes_felt.iterrows():
                    magnitude = row['Magnitude']
                    date = row['Tanggal'].strftime('%d %B %Y')  # Format tanggal DD/MM/YYYY
                    time = row['Waktu (WIB)']  # Asumsikan kolom waktu sudah dalam format string yang benar
                    location = row['Dirasakan']
                    description = row['Keterangan']
                    depth = row['Kedalaman']

                    # Format detail gempa yang dirasakan dengan nomor urut
                    felt_detail = (
                        f"{counter}. Gempabumi dengan magnitudo {magnitude} {location} yang terjadi pada tanggal {date} "
                        f"pukul {time} WIB berlokasi di {description} pada kedalaman {depth} km."
                    )

                    # Menambahkan detail gempa ke dalam dokumen sebagai paragraf terpisah
                    felt_detail_paragraph = document.add_paragraph(felt_detail)
                    felt_detail_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

                    # Meningkatkan counter untuk nomor urut berikutnya
                    counter += 1
            else:
                # Jika tidak ada gempa yang dirasakan, tambahkan keterangan tambahan
                no_felt_detail = document.add_paragraph(" ")
                no_felt_detail.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER


            # Tambahkan section baru dengan orientasi landscape
            new_section = document.add_section(WD_SECTION.NEW_PAGE)
            new_section.orientation = WD_ORIENT.LANDSCAPE

            # Atur ukuran kertas agar landscape sesuai
            new_section.page_width, new_section.page_height = new_section.page_height, new_section.page_width

            # Menentukan margin untuk section baru agar tabel berada di tengah secara vertikal
            section_margin = Cm(2)  # Sesuaikan margin jika perlu
            new_section.top_margin = section_margin
            new_section.bottom_margin = section_margin
            new_section.left_margin = section_margin
            new_section.right_margin = section_margin

            # Menambahkan paragraf baru untuk membungkus tabel
            table_paragraph = document.add_paragraph()
            table_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment paragraf ke tengah

            # Membuat teks keterangan tabel gempabumi
            tabel_keterangan_text = (
                f"Tabel 1. Data gempabumi di Bali dan sekitarnya tanggal {start_date_formatted} – {end_date_formatted}"
            )

            # Menambahkan teks keterangan tabel ke dalam dokumen
            tabel_keterangan_paragraph = document.add_paragraph(tabel_keterangan_text)
            tabel_keterangan_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment menjadi tengah

            # Menambahkan tabel langsung ke dalam paragraf
            table = document.add_table(rows=1, cols=len(earthquake_data.columns))
            table.style = 'Table Grid'

            # Set table alignment to center
            table.alignment = WD_TABLE_ALIGNMENT.CENTER

            # Set column widths to match the text content
            column_widths = {
                "NO.": 1.5,  # NO
                "Tanggal": 2.5,  # TANGGAL
                "Waktu (WIB)": 2.5,  # WAKTU (WIB)
                "Lintang": 2.0,  # LATITUDE
                "Bujur": 2.0,  # LONGITUDE
                "Kedalaman": 2.0,  # Depth
                "Magnitude": 1.5,  # Magnitude
                "Keterangan": 5.0,  # Keterangan
                "Dirasakan": 2.5  # Dirasakan
            }

            # Add header row
            hdr_cells = table.rows[0].cells
            for i, column_name in enumerate(earthquake_data.columns):
                hdr_cells[i].text = column_name
                hdr_cells[i].width = Cm(column_widths.get(column_name, 2.0))  # Set width based on column

            # Add data rows, memformat kolom 'Tanggal' tanpa jam
            for index, row in earthquake_data.iterrows():
                row_cells = table.add_row().cells
                for i, value in enumerate(row):
                    # Jika kolom adalah 'Tanggal', hanya tampilkan tanggal tanpa jam
                    if earthquake_data.columns[i] == 'Tanggal':
                        row_cells[i].text = value.strftime('%d/%m/%Y')  # Format tanggal menjadi DD/MM/YYYY
                    # Jika kolom adalah 'Dirasakan' dan nilai adalah NaN, ganti dengan teks yang sesuai
                    elif earthquake_data.columns[i] == 'Dirasakan':
                        # Ganti NaN dengan string kosong atau teks lain
                        row_cells[i].text = '' if pd.isna(value) else str(value)
                    else:
                        row_cells[i].text = str(value)

                    # Set the width of each data cell, sesuai dengan kolom
                    row_cells[i].width = Cm(column_widths.get(earthquake_data.columns[i], 2.0))

            # Mengatur tabel agar berada di tengah secara horizontal
            table.alignment = WD_TABLE_ALIGNMENT.CENTER
            
            
            # Simpan 'datalengkap.csv' ke folder baru dengan nama 'data gempa (nama folder).xlsx'
            output_file = os.path.join(nama_folder, f'Laporan Gempabumi Periode {nama_folder}.docx')
        
            # Save the document
            document.save(output_file)

        messagebox.showinfo("Cetak Laporan", "Berhasil Mencetak laporan ke file Word")

    def print_press_release(self):
        # Set locale to Indonesian for date formatting
        locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

        # Fungsi untuk memilih file
        def select_file():
            root = tk.Tk()
            root.withdraw()  # Menyembunyikan jendela utama
            file_path = filedialog.askopenfilename(title="Pilih file laporan_gempa.xlsx", filetypes=[("Excel files", "*.xlsx")])
            return file_path

        # Memilih file Excel
        file_path = select_file()
        if not file_path:
            print("Tidak ada file yang dipilih.")
        else:
            # Load the Excel file
            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

            # Tentukan nama folder berdasarkan tanggal awal dan akhir
            if tanggal_awal.month == tanggal_akhir.month:
                nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
            else:
                nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

            # Ambil tanggal awal dan akhir
            start_date = tanggal_awal
            end_date = tanggal_akhir
    
            # Path to images
            magnitude_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_mag.png"
            magnitude_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_mag.png"
            depth_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_depth.png"
            depth_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_depth.png"
            seismic_map_path = f"D:/Seismioto/Pengolahan/{nama_folder}/Peta Seismisitas Periode {nama_folder}.png"
        
            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

            # Calculate the required values
            total_earthquakes = len(earthquake_data)
            min_magnitude = earthquake_data['Magnitude'].min()
            max_magnitude = earthquake_data['Magnitude'].max()

            # Format tanggal dengan nama bulan dalam bahasa Indonesia
            start_date_formatted = start_date.strftime('%d %B %Y')  # contoh format: 01 Januari 2024
            end_date_formatted = end_date.strftime('%d %B %Y')      # contoh format: 05 September 2024

            # Menentukan tanggal awal bulan untuk memulai perhitungan minggu
            start_of_month = start_date.replace(day=1)

            # Hitung nomor minggu relatif terhadap awal bulan
            week_number = (start_date - start_of_month).days // 7 + 1

            # Format bulan dalam bahasa Indonesia
            bulan_formatted = start_date.strftime('%B')

            # Menghitung jumlah kejadian gempabumi total
            total_earthquakes = len(earthquake_data)

            # Menghitung jumlah gempa dengan lat > -9 dan kedalaman < 60 km
            shallow_earthquake_count = earthquake_data[(earthquake_data['Lintang'] > -9) & 
                                                    (earthquake_data['Kedalaman'] < 60)].shape[0]

            # Menghitung jumlah gempa di selatan Pulau Jawa, Bali, dan Lombok
            southern_earthquake_count = total_earthquakes - shallow_earthquake_count

            # Magnitude categories
            magnitude_below_3 = earthquake_data[earthquake_data['Magnitude'] <= 3]
            magnitude_3_to_5 = earthquake_data[(earthquake_data['Magnitude'] > 3) & (earthquake_data['Magnitude'] <= 5)]
            magnitude_above_5 = earthquake_data[earthquake_data['Magnitude'] > 5]

            count_below_3 = len(magnitude_below_3)
            count_3_to_5 = len(magnitude_3_to_5)
            count_above_5 = len(magnitude_above_5)

            percentage_below_3 = int((count_below_3 / total_earthquakes) * 100)
            percentage_3_to_5 = int((count_3_to_5 / total_earthquakes) * 100)
            percentage_above_5 = int((count_above_5 / total_earthquakes) * 100)

            # Magnitude categories
            depth_below_60 = earthquake_data[earthquake_data['Kedalaman'] < 60]
            depth_60_to_300 = earthquake_data[(earthquake_data['Kedalaman'] >= 60) & (earthquake_data['Kedalaman'] <= 300)]
            depth_above_300 = earthquake_data[earthquake_data['Kedalaman'] > 300]

            count_depth_below_60 = len(depth_below_60)
            count_depth_60_to_300 = len(depth_60_to_300)
            count_depth_above_300 = len(depth_above_300)

            percentage_depth_below_60 = int((count_depth_below_60 / total_earthquakes) * 100)
            percentage_depth_60_to_300 = int((count_depth_60_to_300 / total_earthquakes) * 100)
            percentage_depth_above_300 = int((count_depth_above_300 / total_earthquakes) * 100)

            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Create a new Word document
            document = Document()

            # Ukuran kertas F4 (21.0 cm x 33.0 cm)
            f4_width = Cm(21.0)
            f4_height = Cm(33.0)

            # Mengatur ukuran kertas F4 untuk setiap section di dokumen
            for section in document.sections:
                # Mengatur orientasi potrait untuk F4
                section.orientation = WD_ORIENT.PORTRAIT  # Bisa diubah ke WD_ORIENT.LANDSCAPE jika ingin landscape
                section.page_width = f4_width
                section.page_height = f4_height

                # Mengatur margin halaman (optional, bisa disesuaikan)
                section.top_margin = Cm(2.5)     # Marginn atas
                section.bottom_margin = Cm(2.5)  # Marginn bawah
                section.left_margin = Cm(2.5)    # Marginn kiri
                section.right_margin = Cm(2.5)   # Marginn kanan

            # Add Title
            title = document.add_paragraph()
            title_run = title.add_run(f'PRESS RELEASE AKTIVITAS SEISMISITAS                                                                                              WILAYAH BALI DAN SEKITARNYA                                                                                                        PERIODE {start_date_formatted} – {end_date_formatted}')
            title_run.bold = True
            title_run.font.size = Pt(14)
            title_run.font.all_caps = True  # Membuat semua huruf kapital
            title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')

            # Teks Deskriptif Magnitudo dengan perhitungan minggu bulanan
            magnitude_descriptive_text = (
                f"Berdasarkan data Stasiun Geofisika Denpasar selama minggu ke-{week_number} bulan "
                f"{bulan_formatted} {start_date.year}, "
                f"di daerah Bali dan sekitarnya telah terjadi {total_earthquakes} kejadian gempabumi dengan magnitudo bervariasi "
                f"mulai dari M {min_magnitude} sampai M {max_magnitude}. Kejadian gempabumi dengan magnitudo M<3 sejumlah "
                f"{count_below_3} kejadian atau {percentage_below_3}% dari total kejadian gempabumi. Sedangkan untuk M 3 – 5 "
                f"sejumlah {count_3_to_5} kejadian atau {percentage_3_to_5}% dari total kejadian gempabumi "
            )

            # Conditional text for magnitude above 5
            if count_above_5 > 0:
                magnitude_descriptive_text += (
                    f"dan kejadian gempa M>5 sejumlah {count_above_5} kejadian atau {percentage_above_5}% dari total kejadian gempabumi."
                )
            else:
                magnitude_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan magnitudo M>5."

            # Menambahkan paragraf dengan teks deskriptif magnitudo
            magnitude_paragraph = document.add_paragraph(magnitude_descriptive_text)

            # Mengatur alignment paragraf menjadi justify
            magnitude_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            
            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Add space
            document.add_paragraph('')

            # Teks Deskriptif Kedalaman
            depth_descriptive_text = (
                f"Berdasarkan kedalaman, gempabumi yang mendominasi adalah kejadian gempabumi {dominant_category}  dari total kejadian gempabumi. "
                f"Jumlah gempabumi dangkal sebanyak {count_depth_below_60} kejadian gempabumi, "
                f"terdapat sebanyak {count_depth_60_to_300} kejadian gempabumi dengan kedalaman menengah 60 km - 300 km atau {percentage_depth_60_to_300}% dari total kejadian gempabumi, "
            )

            # Conditional text for depth above 300 km
            if count_depth_above_300 > 0:
                depth_descriptive_text += (
                    f"dan kejadian gempa dalam sejumlah {count_depth_above_300} kejadian atau {percentage_depth_above_300}% dari total kejadian gempabumi."
                )
            else:
                depth_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan kedalaman dalam >300 km."

            depth_paragraph = document.add_paragraph(depth_descriptive_text)
            depth_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY   

            # Add space
            document.add_paragraph('')

            # Teks analisis yang diminta
            analisis_text1 = (
                f"Selama satu minggu terakhir, gempabumi yang terjadi di Bali dan sekitarnya dikelompokan berdasarkan sumbernya (Gambar 3):\n"
            )    

            # Menambahkan paragraf analisis ke dalam dokumen
            analisis_paragraph1 = document.add_paragraph(analisis_text1)
            analisis_paragraph1.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

            analisis_text2 = (
                f"1. Sebanyak {southern_earthquake_count} kejadian gempabumi terjadi di selatan Pulau Jawa, Bali, dan Lombok "
                f"yang diperkirakan berasosiasi dengan subduksi Indo Australia-Eurasia.\n"
                f"2. Sebanyak {shallow_earthquake_count} kejadian gempabumi terjadi menyebar di wilayah Pulau Bali, Lombok dan sekitarnya "
                f"diperkirakan berkaitan dengan sesar di belakang busur Flores atau Flores Back Arc Thrust dan aktivitas sesar aktif."
            )

            # Menambahkan paragraf analisis ke dalam dokumen
            analisis_paragraph2 = document.add_paragraph(analisis_text2)
            analisis_paragraph2.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

            # Menghitung jumlah kejadian gempa yang dirasakan
            earthquakes_felt = earthquake_data[earthquake_data['Dirasakan'].notna()]
            total_felt = len(earthquakes_felt)

            # Menyusun narasi pembuka
            if total_felt > 0:
                felt_intro = (
                    f"Selama periode ini terdapat {total_felt} kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )
            else:
                felt_intro = (
                    f"Selama periode ini tidak terdapat kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )

            # Menambahkan narasi pembuka ke dokumen
            felt_intro_paragraph = document.add_paragraph(felt_intro)
            felt_intro_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY    

            # Menyusun detail setiap kejadian gempa yang dirasakan dalam format list dengan nomor urut
            if total_felt > 0:
                # Inisialisasi counter untuk nomor urut
                counter = 1

                for index, row in earthquakes_felt.iterrows():
                    magnitude = row['Magnitude']
                    date = row['Tanggal'].strftime('%d %B %Y')  # Format tanggal DD/MM/YYYY
                    time = row['Waktu (WIB)']  # Asumsikan kolom waktu sudah dalam format string yang benar
                    location = row['Dirasakan']
                    description = row['Keterangan']
                    depth = row['Kedalaman']

                    # Format detail gempa yang dirasakan dengan nomor urut
                    felt_detail = (
                        f"{counter}. Gempabumi dengan magnitudo {magnitude} {location} yang terjadi pada tanggal {date} "
                        f"pukul {time} WIB berlokasi di {description} pada kedalaman {depth} km."
                    )

                    # Menambahkan detail gempa ke dalam dokumen sebagai paragraf terpisah
                    felt_detail_paragraph = document.add_paragraph(felt_detail)
                    felt_detail_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

                    # Meningkatkan counter untuk nomor urut berikutnya
                    counter += 1
            else:
                # Jika tidak ada gempa yang dirasakan, tambahkan keterangan tambahan
                no_felt_detail = document.add_paragraph(" ")
                no_felt_detail.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add Magnitude Bar Chart Image
            try:
                document.add_paragraph().add_run().add_picture(magnitude_bar_chart_path, width=Inches(5.65),) 
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM BATANG MAGNITUDO TIDAK DITEMUKAN').bold = True

            # Add Magnitude Pie Chart Image
            try:
                document.add_paragraph().add_run().add_picture(magnitude_pie_chart_path, width=Inches(3.15))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM LINGKARAN MAGNITUDO TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Magnitudo
            document.add_paragraph('Gambar 1. Histogram gempabumi harian berdasarkan magnitudo (kiri) dan diagram gempabumi berdasarkan magnitudo (kanan)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')

            # Add Depth Bar Chart Image
            try:
                document.add_paragraph().add_run().add_picture(depth_bar_chart_path, width=Inches(5.65))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM BATANG KEDALAMAN TIDAK DITEMUKAN').bold = True

            # Add Depth Pie Chart Image
            try:
                document.add_paragraph().add_run().add_picture(depth_pie_chart_path, width=Inches(3.15))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM LINGKARAN KEDALAMAN TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Kedalaman (akan diisi sesuai format)
            document.add_paragraph('Gambar 2.  Histogram gempabumi harian berdasarkan kedalaman (kiri) dan diagram gempabumi berdasarkan kedalaman (kanan)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')
            
            # Add Seismic Map Image
            try:
                document.add_paragraph().add_run().add_picture(seismic_map_path, width=Inches(6))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('PETA SEISMISITAS TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Peta Seismisitas
            peta_seismisitas_text = (
                f"Gambar 3. Peta seismisitas Bali dan sekitarnya periode {start_date_formatted} - {end_date_formatted}"
            )
            peta_seismisitas_paragraph = document.add_paragraph(peta_seismisitas_text)
            peta_seismisitas_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Tambahkan section baru dengan orientasi landscape
            new_section = document.add_section(WD_SECTION.NEW_PAGE)
            new_section.orientation = WD_ORIENT.LANDSCAPE

            # Atur ukuran kertas agar landscape sesuai
            new_section.page_width, new_section.page_height = new_section.page_height, new_section.page_width

            # Menentukan margin untuk section baru agar tabel berada di tengah secara vertikal
            section_margin = Cm(2)  # Sesuaikan margin jika perlu
            new_section.top_margin = section_margin
            new_section.bottom_margin = section_margin
            new_section.left_margin = section_margin
            new_section.right_margin = section_margin

            # Menambahkan paragraf baru untuk membungkus tabel
            table_paragraph = document.add_paragraph()
            table_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment paragraf ke tengah

            # Membuat teks keterangan tabel gempabumi
            tabel_keterangan_text = (
                f"Tabel 1. Data gempabumi di Bali dan sekitarnya tanggal {start_date_formatted} – {end_date_formatted}"
            )

            # Menambahkan teks keterangan tabel ke dalam dokumen
            tabel_keterangan_paragraph = document.add_paragraph(tabel_keterangan_text)
            tabel_keterangan_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment menjadi tengah

            # Menambahkan tabel langsung ke dalam paragraf
            table = document.add_table(rows=1, cols=len(earthquake_data.columns))
            table.style = 'Table Grid'

            # Set table alignment to center
            table.alignment = WD_TABLE_ALIGNMENT.CENTER

            # Set column widths to match the text content
            column_widths = {
                "NO.": 1.5,  # NO
                "Tanggal": 2.5,  # TANGGAL
                "Waktu (WIB)": 2.5,  # WAKTU (WIB)
                "Lintang": 2.0,  # LATITUDE
                "Bujur": 2.0,  # LONGITUDE
                "Kedalaman": 2.0,  # Depth
                "Magnitude": 1.5,  # Magnitude
                "Keterangan": 5.0,  # Keterangan
                "Dirasakan": 2.5  # Dirasakan
            }

            # Add header row
            hdr_cells = table.rows[0].cells
            for i, column_name in enumerate(earthquake_data.columns):
                hdr_cells[i].text = column_name
                hdr_cells[i].width = Cm(column_widths.get(column_name, 2.0))  # Set width based on column

            # Add data rows, memformat kolom 'Tanggal' tanpa jam
            for index, row in earthquake_data.iterrows():
                row_cells = table.add_row().cells
                for i, value in enumerate(row):
                    # Jika kolom adalah 'Tanggal', hanya tampilkan tanggal tanpa jam
                    if earthquake_data.columns[i] == 'Tanggal':
                        row_cells[i].text = value.strftime('%d/%m/%Y')  # Format tanggal menjadi DD/MM/YYYY
                    # Jika kolom adalah 'Dirasakan' dan nilai adalah NaN, ganti dengan teks yang sesuai
                    elif earthquake_data.columns[i] == 'Dirasakan':
                        # Ganti NaN dengan string kosong atau teks lain
                        row_cells[i].text = '' if pd.isna(value) else str(value)
                    else:
                        row_cells[i].text = str(value)

                    # Set the width of each data cell, sesuai dengan kolom
                    row_cells[i].width = Cm(column_widths.get(earthquake_data.columns[i], 2.0))

            # Mengatur tabel agar berada di tengah secara horizontal
            table.alignment = WD_TABLE_ALIGNMENT.CENTER
            
            
            # Simpan 'datalengkap.csv' ke folder baru dengan nama 'data gempa (nama folder).xlsx'
            output_file = os.path.join(nama_folder, f'Press Release Gempabumi Periode {nama_folder}.docx')
        
            # Save the document
            document.save(output_file)

        messagebox.showinfo("Cetak Laporan", "Berhasil Mencetak Press Release")

    def print_description(self):
        # Set locale to Indonesian for date formatting
        locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

        # Fungsi untuk memilih file
        def select_file():
            root = tk.Tk()
            root.withdraw()  # Menyembunyikan jendela utama
            file_path = filedialog.askopenfilename(title="Pilih file laporan_gempa.xlsx", filetypes=[("Excel files", "*.xlsx")])
            return file_path

        # Memilih file Excel
        file_path = select_file()
        if not file_path:
            print("Tidak ada file yang dipilih.")
        else:
            # Load the Excel file
            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

            # Ambil tanggal awal dan akhir
            start_date = tanggal_awal
            end_date = tanggal_akhir
            
            # Format tanggal untuk digunakan dalam nama folder     

            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

            # Tentukan nama folder berdasarkan tanggal awal dan akhir
            if tanggal_awal.month == tanggal_akhir.month:
                nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
            else:
                nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

            # Calculate the required values
            total_earthquakes = len(earthquake_data)
            min_magnitude = earthquake_data['Magnitude'].min()
            max_magnitude = earthquake_data['Magnitude'].max()

            # Date range
            start_date = earthquake_data['Tanggal'].min()
            end_date = earthquake_data['Tanggal'].max()
            #date1 = earthquake_data['Tanggal']

            # Format tanggal dengan nama bulan dalam bahasa Indonesia
            start_date_formatted = start_date.strftime('%d %B %Y')  # contoh format: 01 Januari 2024
            end_date_formatted = end_date.strftime('%d %B %Y')      # contoh format: 05 September 2024
            #date_formatted=date1.strftime('%d %B %Y')
            #print(date_formatted)

            # Menentukan tanggal awal bulan untuk memulai perhitungan minggu
            start_of_month = start_date.replace(day=1)

            # Hitung nomor minggu relatif terhadap awal bulan
            week_number = (start_date - start_of_month).days // 7 + 1

            # Format bulan dalam bahasa Indonesia
            bulan_formatted = start_date.strftime('%B')

            # Menghitung jumlah kejadian gempabumi total
            total_earthquakes = len(earthquake_data)

            # Menghitung jumlah gempa dengan lat < -9 dan kedalaman < 60 km
            shallow_earthquake_count = earthquake_data[(earthquake_data['Lintang'] < -9) & 
                                                    (earthquake_data['Kedalaman'] < 60)].shape[0]

            # Menghitung jumlah gempa di selatan Pulau Jawa, Bali, dan Lombok
            southern_earthquake_count = total_earthquakes - shallow_earthquake_count

            # Magnitude categories
            magnitude_below_3 = earthquake_data[earthquake_data['Magnitude'] <= 3]
            magnitude_3_to_5 = earthquake_data[(earthquake_data['Magnitude'] > 3) & (earthquake_data['Magnitude'] <= 5)]
            magnitude_above_5 = earthquake_data[earthquake_data['Magnitude'] > 5]

            count_below_3 = len(magnitude_below_3)
            count_3_to_5 = len(magnitude_3_to_5)
            count_above_5 = len(magnitude_above_5)

            percentage_below_3 = int((count_below_3 / total_earthquakes) * 100)
            percentage_3_to_5 = int((count_3_to_5 / total_earthquakes) * 100)
            percentage_above_5 = int((count_above_5 / total_earthquakes) * 100)

            # Magnitude categories
            depth_below_60 = earthquake_data[earthquake_data['Kedalaman'] < 60]
            depth_60_to_300 = earthquake_data[(earthquake_data['Kedalaman'] >= 60) & (earthquake_data['Kedalaman'] <= 300)]
            depth_above_300 = earthquake_data[earthquake_data['Kedalaman'] > 300]

            count_depth_below_60 = len(depth_below_60)
            count_depth_60_to_300 = len(depth_60_to_300)
            count_depth_above_300 = len(depth_above_300)

            percentage_depth_below_60 = int((count_depth_below_60 / total_earthquakes) * 100)
            percentage_depth_60_to_300 = int((count_depth_60_to_300 / total_earthquakes) * 100)
            percentage_depth_above_300 = int((count_depth_above_300 / total_earthquakes) * 100)

            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Create a new Word document
            document = Document()

            # Ukuran kertas F4 (21.0 cm x 33.0 cm)
            f4_width = Cm(21.0)
            f4_height = Cm(33.0)

            # Mengatur ukuran kertas F4 untuk setiap section di dokumen
            for section in document.sections:
                # Mengatur orientasi potrait untuk F4
                section.orientation = WD_ORIENT.PORTRAIT  # Bisa diubah ke WD_ORIENT.LANDSCAPE jika ingin landscape
                section.page_width = f4_width
                section.page_height = f4_height

                # Mengatur margin halaman (optional, bisa disesuaikan)
                section.top_margin = Cm(2.5)     # Marginn atas
                section.bottom_margin = Cm(2.5)  # Marginn bawah
                section.left_margin = Cm(2.5)    # Marginn kiri
                section.right_margin = Cm(2.5)   # Marginn kanan

            # Add Subtitle
            subtitle = document.add_paragraph()
            subtitle_run = subtitle.add_run('Informasi Gempabumi di Wilayah Bali dan Sekitarnya')
            subtitle_run.font.size = Pt(12)
            subtitle.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add Date Period
            period = document.add_paragraph()
            period_run = period.add_run(f'Periode {nama_folder}')
            period_run.font.size = Pt(12)
            period.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')

            # Teks Deskriptif Magnitudo dengan perhitungan minggu bulanan
            magnitude_descriptive_text = (
                f"Berdasarkan data Stasiun Geofisika Denpasar selama minggu ke-{week_number} bulan "
                f"{bulan_formatted} {start_date.year}, "
                f"di daerah Bali dan sekitarnya telah terjadi {total_earthquakes} kejadian gempabumi dengan magnitudo bervariasi "
                f"mulai dari M {min_magnitude} sampai M {max_magnitude}. Kejadian gempabumi dengan magnitudo M<3 sejumlah "
                f"{count_below_3} kejadian atau {percentage_below_3}% dari total kejadian gempabumi. Sedangkan untuk M 3 – 5 "
                f"sejumlah {count_3_to_5} kejadian atau {percentage_3_to_5}% dari total kejadian gempabumi "
            )

            # Conditional text for magnitude above 5
            if count_above_5 > 0:
                magnitude_descriptive_text += (
                    f"dan kejadian gempa M>5 sejumlah {count_above_5} kejadian atau {percentage_above_5}% dari total kejadian gempabumi."
                )
            else:
                magnitude_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan magnitudo M>5."

            # Menambahkan paragraf dengan teks deskriptif magnitudo
            magnitude_paragraph = document.add_paragraph(magnitude_descriptive_text)

            # Mengatur alignment paragraf menjadi justify
            magnitude_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Teks Deskriptif Kedalaman
            depth_descriptive_text = (
                f"Berdasarkan kedalaman, gempabumi yang mendominasi adalah kejadian gempabumi {dominant_category}  dari total kejadian gempabumi. "
                f"Jumlah gempabumi dangkal sebanyak {count_depth_below_60} kejadian gempabumi, "
                f"terdapat sebanyak {count_depth_60_to_300} kejadian gempabumi dengan kedalaman menengah 60 km - 300 km atau {percentage_depth_60_to_300}% dari total kejadian gempabumi, "
            )

            # Conditional text for depth above 300 km
            if count_depth_above_300 > 0:
                depth_descriptive_text += (
                    f"dan kejadian gempa dalam sejumlah {count_depth_above_300} kejadian atau {percentage_depth_above_300}% dari total kejadian gempabumi."
                )
            else:
                depth_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan kedalaman dalam >300 km."

            depth_paragraph = document.add_paragraph(depth_descriptive_text)
            depth_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Menghitung jumlah kejadian gempa yang dirasakan
            earthquakes_felt = earthquake_data[earthquake_data['Dirasakan'].notna()]
            total_felt = len(earthquakes_felt)

            # Menyusun narasi pembuka
            if total_felt > 0:
                felt_intro = (
                    f"Selama periode ini terdapat {total_felt} kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )
            else:
                felt_intro = (
                    f"Selama periode ini tidak terdapat kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )

            # Menambahkan narasi pembuka ke dokumen
            felt_intro_paragraph = document.add_paragraph(felt_intro)
            felt_intro_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Menyusun detail setiap kejadian gempa yang dirasakan dalam format list dengan nomor urut
            if total_felt > 0:
                # Inisialisasi counter untuk nomor urut
                counter = 1

                for index, row in earthquakes_felt.iterrows():
                    magnitude = row['Magnitude']
                    date = row['Tanggal'].strftime('%d %B %Y')  # Format tanggal DD/MM/YYYY
                    time = row['Waktu (WIB)']  # Asumsikan kolom waktu sudah dalam format string yang benar
                    location = row['Dirasakan']
                    description = row['Keterangan']
                    depth = row['Kedalaman']

                    # Format detail gempa yang dirasakan dengan nomor urut
                    felt_detail = (
                        f"{counter}. Gempabumi dengan magnitudo {magnitude} {location} yang terjadi pada tanggal {date} "
                        f"pukul {time} WIB berlokasi di {description} pada kedalaman {depth} km."
                    )

                    # Menambahkan detail gempa ke dalam dokumen sebagai paragraf terpisah
                    felt_detail_paragraph = document.add_paragraph(felt_detail)
                    felt_detail_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

                    # Meningkatkan counter untuk nomor urut berikutnya
                    counter += 1
            else:
                # Jika tidak ada gempa yang dirasakan, tambahkan keterangan tambahan
                no_felt_detail = document.add_paragraph("Tidak ada rincian gempabumi yang dirasakan selama periode ini.")
                no_felt_detail.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')
            
            document.add_paragraph("Halo sobat BMKG Denpasar! Apa kabar? Semoga sehat ya🤗\n" 
                                f"Berikut kami informasikan aktivitas kegempaan wilayah Bali dan sekitarnya periode {nama_folder}."
                                "Pada minggu ini, aktivitas kegempaan didominasi oleh gempa magnitudo M < 3 dengan kedalaman dangkal < 60 km."
                                f"Pada minggu ini terdapat {total_earthquakes} kejadian gempabumi." \
                                "\n" \
                                "\n" \
                                f"Jumat, {end_date_formatted}.\n"
                                ".\n"
                                ".\n"
                                ".\n"
                                "Hello BMKG Denpasar friends! How are you? Hope you are healthy🤗\n" 
                                f"Here we provide information on earthquake activity in the Bali area and its surroundings for the period {start_date_formatted} – {end_date_formatted}."
                                "This week, seismic activity was dominated by earthquakes with a magnitude of M < 3 with a shallow depth of < 60 km."
                                f"This week there are {total_earthquakes} earthquake event. Greetings of health and enthusiasm." \
                                "\n" \
                                "\n" \
                                f"Friday, {end_date_formatted}.\n"
                                "#stageofdenpasar #dewata #dedikasi #wibawa #tanggap #akuntabel #siapuntukselamat #kukul #kentongan #linuh")

            # Add space
            document.add_paragraph('')
            
            document.add_paragraph("Halo sobat BMKG Denpasar! Apa kabar? Semoga sehat ya🤗.\n"
                                f"Berikut kami informasikan aktivitas kegempaan wilayah Bali dan sekitarnya periode {nama_folder}.\n"
                                "Salam sehat dan semangat. \n"
                                "#stageofdenpasar #dewata #dedikasi #wibawa #tanggap #akuntabel #siapuntukselamat ")


            # Add space
            document.add_paragraph('')

            # Tambah penerima
            document.add_paragraph("Yth.\n1. Ibu Deputi Bidang Geofisika\n"
                                "2. Bapak Direktur Gempabumi dan Tsunami\n"
                                "3. Bapak Direktur Seismologi Teknik, Geofisika Potensial dan Tanda Waktu\n"
                                "4. Bapak Kepala Balai Wilayah III\n")

            # Tambah salam pembuka
            document.add_paragraph("Dengan hormat,\nBerikut kami sampaikan sebagai berikut:")

            # Tambah poin pertama
            document.add_paragraph(f"1. Infografis, videografis dan data excel seismisitas wilayah Bali periode {nama_folder}. "
                                "Laporan lengkap kami kirimkan ke email: zona.potensi@gmail.com")

            # Tambah daftar link media sosial
            document.add_paragraph("\nBerikut terlampir link media sosial infografis dan videografis seismisitas wilayah Bali dan sekitarnya dari Stasiun Geofisika Denpasar:")

            social_links = [
                ("1. Link Instagram", "Videografis dan Infografis: https://www.instagram.com/bmkg_denpasar"),
                ("2. Link Facebook", "Videografis dan Infografis: https://web.facebook.com/BMKGDenpasar"),
                ("3. Link X", "Videografis dan Infografis: https://x.com/BMKG_Denpasar"),
            ]

            for title, link in social_links:
                p = document.add_paragraph()
                p.add_run(title).bold = True
                p.add_run("\n" + link)

            # Tambah penutup
            document.add_paragraph("\nDemikian, mohon berkenan dan terima kasih.")
            document.add_paragraph("Hormat kami,\nBMKG Stasiun Geofisika Denpasar")

            # Tambah kontak
            document.add_paragraph("\nKontak Stasiun Geofisika Denpasar:")
            contacts = [
                "📍 Alamat: Jl. Pulau Tarakan No. 1 Sanglah, Denpasar-Bali",
                "📞 Telepon: (0361) 226157",
                "🌐 Web: stageof-bali.bmkg.go.id",
                "📧 Email: stageof.denpasar@bmkg.go.id"
            ]

            for contact in contacts:
                document.add_paragraph(contact)

            # Tambah kalimat penutup
            document.add_paragraph("\nDemikian yang dapat kami sampaikan.\n\nTerima kasih.")

        # Simpan 'datalengkap.csv' ke folder baru dengan nama 'data gempa (nama folder).xlsx'
            output_file = os.path.join(nama_folder, f'Deskripsi info seismisitas website {nama_folder}.docx')
        
            # Save the document
            document.save(output_file)
            
        messagebox.showinfo("Cetak Deskripsi", "Berhasil Mencetak deskripsi")

if __name__ == "__main__":
    root = tk.Tk()
    app = EarthquakeApp(root)
    root.mainloop()


In [4]:
import numpy as np
import pygmt
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
import tkinter as tk
import shutil  # Untuk menyalin file
import locale
from datetime import datetime, timedelta
from docx import Document
from docx.shared import Pt, Inches
from docx.shared import Cm
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.section import WD_SECTION, WD_ORIENT
from openpyxl import load_workbook
from tkinter import filedialog
from tkinter import messagebox
from tkinter import ttk
from tkcalendar import DateEntry 

# Variabel global untuk menyimpan file yang dipilih
selected_files = []

class EarthquakeApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Otomatisasi Informasi Gempabumi Mingguan")
        self.root.geometry("800x600")
        self.root.resizable(False, False)
        self.root.configure(background='#e6f2ff')

        self.cal_start = None
        self.cal_end = None
        self.start_date_var = tk.StringVar(value=datetime.now().strftime("%d/%m/%Y"))
        self.end_date_var = tk.StringVar(value=datetime.now().strftime("%d/%m/%Y"))

        self.style = ttk.Style()
        self.style.configure('TFrame', background='#e6f2ff')
        self.style.configure('Left.TFrame', background='#cce6ff')
        self.style.configure('TLabel', background='#e6f2ff', font=('Arial', 10), foreground='#333')
        self.style.configure('Title.TLabel', font=('Arial', 14, 'bold'), background='#e6f2ff', foreground='#003366')
        self.style.configure('TButton', font=('Arial', 10), padding=5, foreground='black')
        self.style.map('TButton', background=[('active', '#80c1ff')])
        self.style.configure('Print.TButton', background='#4da6ff', foreground='black')  # changed to black

        self.main_container = ttk.Frame(root, style='TFrame')
        self.main_container.pack(fill=tk.BOTH, expand=True)

        self.left_panel = ttk.Frame(self.main_container, width=200, padding=10, style='Left.TFrame')
        self.left_panel.pack(side=tk.LEFT, fill=tk.Y)

        self.right_panel = ttk.Frame(self.main_container, padding=10, style='TFrame')
        self.right_panel.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

        title_label = ttk.Label(self.right_panel, text="Otomatisasi Informasi Gempabumi Mingguan", style='Title.TLabel')
        title_label.pack(pady=(0, 20))

        self.instruction_text = tk.Text(self.right_panel, height=8, wrap=tk.WORD, font=('Arial', 10), bg='#ffffff', fg='#000000')
        self.instruction_text.pack(fill=tk.X, padx=5, pady=(0, 20))
        self.instruction_text.insert(tk.END, "Petunjuk:\n- Silakan pilih file gempa.\n- Atur tanggal awal dan akhir.\n- Klik 'Proses' untuk memulai analisis.\n- Gunakan tombol cetak untuk membuat laporan atau peta.\n")

        self.create_buttons()

    def create_buttons(self):
        ttk.Button(self.left_panel, text="Pilih File", command=self.select_file).pack(pady=5)

        ttk.Button(self.left_panel, text="Pilih Tanggal", command=self.show_calendar_popup).pack(pady=5)
        ttk.Label(self.left_panel, text="Tanggal Awal:", style='TLabel').pack()
        ttk.Label(self.left_panel, textvariable=self.start_date_var).pack()
        ttk.Label(self.left_panel, text="Tanggal Akhir:", style='TLabel').pack()
        ttk.Label(self.left_panel, textvariable=self.end_date_var).pack()

        ttk.Button(self.left_panel, text="Proses", command=self.process_data).pack(pady=20)

        print_buttons = [
            ("Cetak Analisis xlsx", self.print_analysis),
            ("Cetak Peta", self.print_map),
            ("Cetak Laporan", self.print_report),
            ("Cetak Press Release", self.print_press_release),
            ("Cetak Deskripsi", self.print_description)
        ]

        for text, command in print_buttons:
            ttk.Button(self.left_panel, text=text, command=command, style='Print.TButton').pack(pady=3)

        ttk.Button(self.left_panel, text="Keluar", command=self.force_exit).pack(side=tk.BOTTOM, pady=20)

    def show_calendar_popup(self):
        self.hide_calendars()

        top = tk.Toplevel(self.root)
        top.title("Pilih Tanggal Awal dan Akhir")
        top.configure(background="#f9fcff")

        ttk.Label(top, text="Tanggal Awal", background="#f9fcff").pack(pady=(5, 0))
        cal_start = Calendar(top, selectmode='day', date_pattern='dd/mm/yyyy')
        cal_start.pack(pady=5)

        ttk.Label(top, text="Tanggal Akhir", background="#f9fcff").pack(pady=(10, 0))
        cal_end = Calendar(top, selectmode='day', date_pattern='dd/mm/yyyy')
        cal_end.pack(pady=5)

        def set_dates():
            self.start_date_var.set(cal_start.get_date())
            self.end_date_var.set(cal_end.get_date())
            top.destroy()

        ttk.Button(top, text="Pilih", command=set_dates).pack(pady=10)
        self.cal_start = top

    def hide_calendars(self):
        if self.cal_start:
            self.cal_start.destroy()
            self.cal_start = None
        if self.cal_end:
            self.cal_end.destroy()
            self.cal_end = None

    def select_file(self):
        global selected_files
        selected_files = filedialog.askopenfilenames(title="Pilih File", filetypes=(("Excel Files", "*.xlsx"), ("Semua File", "*.*")))
        if selected_files:
            messagebox.showinfo("File Terpilih", f"Jumlah file terpilih: {len(selected_files)}")

    def standardize_months(text):
        month_map = {
            'Jan': '01', 'January': '01', 'Januari': '01',
            'Feb': '02', 'February': '02', 'Februari': '02',
            'Mar': '03', 'March': '03', 'Maret': '03',
            'Apr': '04', 'April': '04',
            'May': '05', 'Mei': '05',
            'Jun': '06', 'June': '06', 'Juni': '06',
            'Jul': '07', 'July': '07', 'Juli': '07',
            'Aug': '08', 'August': '08', 'Agustus': '08',
            'Sep': '09', 'September': '09',
            'Oct': '10', 'October': '10', 'Oktober': '10',
            'Nov': '11', 'November': '11',
            'Dec': '12', 'December': '12', 'Desember': '12'
        }
        
        pattern = r'(\d{1,2})-([a-zA-Z]+)-(\d{2})'
        
        def replace_month(match):
            day = match.group(1)
            month = match.group(2)
            year = match.group(3)
            month_num = month_map.get(month.title(), month)  # Default to original if not found
            return f"{day}-{month_num}-{year}"
        
        return re.sub(pattern, replace_month, text)

    def process_data(self):
        global selected_files, tanggal_awal, tanggal_akhir, nama_folder
        locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

        start_date_str = self.start_date_var.get()
        end_date_str = self.end_date_var.get()

        tanggal_awal = pd.to_datetime(start_date_str, format='%d/%m/%Y')
        tanggal_akhir = pd.to_datetime(end_date_str, format='%d/%m/%Y')

        if not selected_files:
            messagebox.showerror("Error", "Pilih file terlebih dahulu!")
            return None, None

        column_name = 'Info Gempa' 

        def parse_text(text):
            # Standardize month names first
            standardized_text = standardize_months(text)
            
            pattern = r"Mag:(?P<magnitude>\d+\.\d+) , (?P<date>\d{2}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) WIB, Lok:(?P<latitude>\d+\.\d+) LS,(?P<longitude>\d+\.\d+) BT \((?P<description>.+?)\), Kedlmn:(?P<depth>\d+) Km"
            match = re.search(pattern, standardized_text)
            if match:
                result = match.groupdict()
                result['latitude'] = '-' + result['latitude']  # Tambahkan tanda negatif untuk LS
                return result
            return None

        combined_data = pd.DataFrame()
        
        for file in selected_files:
            df = pd.read_excel(file, header=1)
            # Apply standardization before parsing
            df[column_name] = df[column_name].astype(str).apply(self.standardize_months)
            parsed_data = df[column_name].dropna().apply(parse_text).apply(pd.Series)
            parsed_data = parsed_data.rename(columns={
                'date': 'Tanggal',
                'time': 'Waktu (WIB)',
                'latitude': 'Lintang',
                'longitude': 'Bujur',
                'magnitude': 'Magnitude',
                'depth': 'Kedalaman',
                'description': 'Keterangan'
            })

            parsed_data['Dirasakan'] = ''
            parsed_data['Tanggal'] = pd.to_datetime(parsed_data['Tanggal'], format='%d-%m-%y', errors='coerce')

            start_longitude = 113.21
            end_longitude = 117.31

            mask = (
                (parsed_data['Tanggal'] >= tanggal_awal) & 
                (parsed_data['Tanggal'] <= tanggal_akhir) &
                (parsed_data['Bujur'].astype(float) >= start_longitude) & 
                (parsed_data['Bujur'].astype(float) <= end_longitude)
            )
            filtered_data = parsed_data[mask]
            combined_data = pd.concat([combined_data, filtered_data])

        combined_data.to_csv('dataspk.csv', index=False)
        hasil_data = pd.read_csv('dataspk.csv')

        existing_file = 'dirasakan.xlsx'
        if not os.path.exists(existing_file):
            messagebox.showerror("Error", f"File '{existing_file}' tidak ditemukan!")
            return

        existing_data = pd.read_excel(existing_file)

        # Konversi tipe data kolom kunci agar konsisten
        key_columns = ['Tanggal', 'Waktu (WIB)', 'Magnitude', 'Lintang', 'Bujur', 'Kedalaman']
        for col in key_columns:
            if col in existing_data.columns:
                if col == 'Tanggal':
                    existing_data[col] = pd.to_datetime(existing_data[col], errors='coerce')
                else:
                    existing_data[col] = existing_data[col].astype(str)

            if col in hasil_data.columns:
                if col == 'Tanggal':
                    hasil_data[col] = pd.to_datetime(hasil_data[col], errors='coerce')
                else:
                    hasil_data[col] = hasil_data[col].astype(str)

        merged_data = pd.merge(existing_data, hasil_data, on=key_columns, how='outer', suffixes=('', '_new'))

        for col in hasil_data.columns:
            if col not in key_columns:
                merged_data[col] = merged_data[col + '_new'].combine_first(merged_data[col])
                merged_data.drop(columns=[col + '_new'], inplace=True)

        merged_data['Datetime'] = pd.to_datetime(merged_data['Tanggal'].astype(str) + ' ' + merged_data['Waktu (WIB)'], errors='coerce')
        merged_data = merged_data.sort_values(by='Datetime').drop(columns=['Datetime'])
        merged_data['Tanggal'] = pd.to_datetime(merged_data['Tanggal']).dt.date

        column_order = ['No', 'Tanggal', 'Waktu (WIB)', 'Lintang', 'Bujur', 'Kedalaman', 'Magnitude', 'Keterangan', 'Dirasakan']
        merged_data.insert(0, 'No', range(1, len(merged_data) + 1))
        merged_data = merged_data[column_order]
        
        merged_data.to_csv('datalengkap.csv', index=False)

        if tanggal_awal.month == tanggal_akhir.month:
            nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
        else:
            nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

        if not os.path.exists(nama_folder):
            os.makedirs(nama_folder)

        output_file = os.path.join(nama_folder, f'Data Gempabumi Periode {nama_folder}.xlsx')
        merged_data.to_excel(output_file, index=False)

        messagebox.showinfo("Proses", f"Data berhasil diproses dan disimpan ke:\n1. {output_file}\n2. datalengkap.csv di direktori script.")

    def print_analysis(self):
        def read_csv_and_analyze(tanggal_awal, tanggal_akhir):
        
            # Step 1: Read CSV file
            file_path = "datalengkap.csv"
            data = pd.read_csv(file_path)
            
            # Konversi kolom Tanggal ke datetime dan ambil hanya tanggalnya
            data['Tanggal'] = pd.to_datetime(data['Tanggal']).dt.date
            tanggal_awal = pd.to_datetime(tanggal_awal).date()
            tanggal_akhir = pd.to_datetime(tanggal_akhir).date()
            
            # Filter data berdasarkan tanggal yang dipilih
            data = data[(data['Tanggal'] >= tanggal_awal) & (data['Tanggal'] <= tanggal_akhir)]
            all_dates = pd.date_range(start=tanggal_awal, end=tanggal_akhir, freq='D')
            periode = all_dates.strftime('%Y-%m-%d')  # Format tanggal sebagai string

            # Drop the 'Magnitude Category' and 'Depth Category' columns
            output_data = data.drop(columns=['Magnitude Category', 'Depth Category'], errors='ignore')

            # Create magnitude and depth categories
            categories_magnitude = ['M<3', '3≤M<5', 'M≥5']
            data['Magnitude'] = pd.to_numeric(data['Magnitude'], errors='coerce')
            data['Magnitude Category'] = pd.cut(data['Magnitude'], bins=[0, 3, 5, float('inf')], labels=categories_magnitude)

            categories_depth = ['D≤60 km', '60<D≤300 km', 'D>300 km']
            data['Kedalaman'] = pd.to_numeric(data['Kedalaman'], errors='coerce')
            data['Depth Category'] = pd.cut(data['Kedalaman'], bins=[0, 60, 300, float('inf')], labels=categories_depth)      
            
            # Group data by day and categories for analysis
            # magnitude_summary = (data.groupby([data['Tanggal'], 'Magnitude Category']).size().unstack(fill_value=0).reindex(index=all_dates, fill_value=0).reindex(columns=categories_magnitude, fill_value=0))
            # depth_summary = (data.groupby([data['Tanggal'], 'Depth Category']).size().unstack(fill_value=0).reindex(index=all_dates, fill_value=0).reindex(columns=categories_depth, fill_value=0))
            
            magnitude_summary = (data.groupby([data['Tanggal'], 'Magnitude Category'])
                                .size()
                                .unstack(fill_value=0)
                                .reindex(index=(pd.date_range(start=tanggal_awal, end=tanggal_akhir, freq='D')), fill_value=0)
                                .reindex(columns=categories_magnitude, fill_value=0))

            depth_summary = (data.groupby([data['Tanggal'], 'Depth Category'])
                            .size()
                            .unstack(fill_value=0)
                            .reindex(index=(pd.date_range(start=tanggal_awal, end=tanggal_akhir, freq='D')), fill_value=0)
                            .reindex(columns=categories_depth, fill_value=0))

            # Add totals row and columns for 'Gempa Dirasakan' and 'Gempa Merusak'
            magnitude_summary['Jumlah Total'] = magnitude_summary.sum(axis=1)
            magnitude_summary['Gempa Dirasakan'] = 0
            magnitude_summary['Gempa Merusak'] = 0
            magnitude_summary.loc['Jumlah Gempa'] = magnitude_summary.sum()

            depth_summary['Jumlah Total'] = depth_summary.sum(axis=1)
            depth_summary['Gempa Dirasakan'] = 0
            depth_summary['Gempa Merusak'] = 0
            depth_summary.loc['Jumlah Gempa'] = depth_summary.sum()

            # Mengisi kolom 'Gempa Dirasakan' jika kolom 'Dirasakan' terisi
            for index, row in data.iterrows():
                if pd.notna(row['Dirasakan']) and row['Dirasakan'].strip() != '':
                    if row['Tanggal'] in magnitude_summary.index:
                        magnitude_summary.at[row['Tanggal'], 'Gempa Dirasakan'] += 1
                    if row['Tanggal'] in depth_summary.index:
                        depth_summary.at[row['Tanggal'], 'Gempa Dirasakan'] += 1

            magnitude_summary.at['Jumlah Gempa', 'Gempa Dirasakan'] = magnitude_summary['Gempa Dirasakan'].sum()
            depth_summary.at['Jumlah Gempa', 'Gempa Dirasakan'] = depth_summary['Gempa Dirasakan'].sum()

            if tanggal_awal.month == tanggal_akhir.month:
                nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
            else:
                nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

            # Untuk menangani row 'Jumlah Gempa' yang bukan tanggal
            magnitude_summary.index = [x.strftime('%Y-%m-%d') if isinstance(x, pd.Timestamp) else x for x in magnitude_summary.index]
            depth_summary.index = [x.strftime('%Y-%m-%d') if isinstance(x, pd.Timestamp) else x for x in depth_summary.index]

            # Save the analysis to Excel
            output_file = os.path.join(nama_folder, f'Analisis Gempabumi Periode {nama_folder}.xlsx')
            with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
                output_data.to_excel(writer, sheet_name='Earthquake Data', index=False)
                magnitude_summary.to_excel(writer, sheet_name='Magnitude Summary')
                depth_summary.to_excel(writer, sheet_name='Depth Summary')
                plot_charts(magnitude_summary, depth_summary, writer, nama_folder, tanggal_awal, tanggal_akhir)

            # Autofit columns
            autofit_columns(output_file, 'Earthquake Data')
            autofit_columns(output_file, 'Magnitude Summary')
            autofit_columns(output_file, 'Depth Summary')

            messagebox.showinfo("Info", f"Analisis berhasil disimpan di:\n{output_file}")

        def plot_charts(magnitude_summary, depth_summary, writer, nama_folder, tanggal_awal, tanggal_akhir,):
            # Set the size for both bar and pie charts
            bar_chart_size = (16, 10)
            pie_chart_size = (5, 5)

            # ================== MAGNITUDE BAR CHART ==================
            categories = ['M<3', '3≤M<5', 'M≥5']
            colors = ['green', 'yellow', 'red']
            
            # Remove total row
            magnitude_summary_without_total = magnitude_summary.drop(index='Jumlah Gempa', errors='ignore')
            
            # Reindex with all dates and fill missing with 0
            magnitude_summary_without_total.index = pd.to_datetime(magnitude_summary_without_total.index)
            magnitude_summary_without_total.index = magnitude_summary_without_total.index.strftime('%d-%m-%Y')

            # Plot setup
            fig, ax = plt.subplots(figsize=(12, 6))
            bar_width = 0.25
            x = np.arange(len(magnitude_summary_without_total.index))

            # Plot each category
            for i, category in enumerate(categories):
                ax.bar(x + i * bar_width, 
                    magnitude_summary_without_total[category], 
                    bar_width, 
                    label=category, 
                    color=colors[i], 
                    edgecolor='black')

            # Chart formatting
            ax.set_title('Frekuensi Gempa Berdasarkan Magnitudo',fontsize=14, pad=20)
            ax.set_xlabel('Tanggal', fontsize=12)
            ax.set_ylabel('Jumlah Kejadian', fontsize=12)
            
            # Set x-ticks to show all dates
            ax.set_xticks(x + bar_width)
            
            # Update x-tick labels dengan format baru
            ax.set_xticklabels(magnitude_summary_without_total.index, rotation=45, ha='right', fontsize=8)
            
            ax.legend(title='Kategori Magnitudo', fontsize=10)
            ax.grid(axis='y', linestyle='--', alpha=0.7)

            # Add value labels
            for p in ax.patches:
                height = p.get_height()
                if height > 0:  # Hanya tampilkan label jika nilai > 0
                    ax.annotate(f'{int(height)}', 
                            (p.get_x() + p.get_width() / 2, height),
                            ha='center', va='bottom', fontsize=8)

            plt.tight_layout()
            plt.savefig(os.path.join(nama_folder, 'diagbat_mag.png'), dpi=300)
            plt.close()

            # Insert to Excel
            worksheet_mag = writer.sheets['Magnitude Summary']
            worksheet_mag.insert_image('A10', os.path.join(nama_folder, 'diagbat_mag.png'), 
                                    {'x_scale': 1.2, 'y_scale': 1.4})

            # Buat diagram lingkaran magnitude
            magnitude_total = magnitude_summary.loc['Jumlah Gempa', ['M<3', '3≤M<5', 'M≥5']]
            labels = ['M<3', '3≤M<5', 'M≥5']
            colors = ['green', 'yellow', 'red']

            # Membuat figure dan pie chart
            fig, ax = plt.subplots(figsize=(6, 6))

            # Fungsi untuk menampilkan persentase dan jumlah kejadian di setiap segmen
            def func(pct, allvals):
                absolute = int(np.round(pct/100.*np.sum(allvals)))
                if absolute > 0:
                    return f'{pct:.1f}%\n({absolute} kejadian)'
                else:
                    return ''  # Jika tidak ada kejadian, label tidak ditampilkan

            # Membuat pie chart
            wedges, texts, autotexts = ax.pie(
                magnitude_total,
                labels=labels,
                colors=colors,
                startangle=85,
                autopct=lambda pct: func(pct, magnitude_total),  # Menampilkan persentase dan jumlah kejadian
                pctdistance=0.875,  # Jarak persentase dari pusat pie
                labeldistance=9  # Jarak label dari pusat pie (aslinya 1.17)
            )

            # Mengatur ukuran font dan latar belakang putih untuk label dan persentase
            for text in texts:
                text.set_fontsize(12)
                text.set_bbox(dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.2'))

            for autotext in autotexts:
                autotext.set_fontsize(12)
                autotext.set_bbox(dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.2'))

            plt.title('Distribusi Magnitudo', fontsize=11)
            ax.legend(title='Kategori Magnitudo', fontsize=7, title_fontsize='9', loc='lower right')
            plt.tight_layout()

            plt.savefig(os.path.join(nama_folder, 'diaglingk_mag.png'))

            # Insert the magnitude pie chart into Excel
            worksheet_mag.insert_image('W10', os.path.join(nama_folder, 'diaglingk_mag.png'), {'x_scale': 1.2, 'y_scale': 1.2})
            
            # ================== DEPTH BAR CHART ==================
            categories_depth = ['D≤60 km', '60<D≤300 km', 'D>300 km']
            colors_depth = ['red', 'yellow', 'green']
            
        # Remove total row
            depth_summary_without_total = depth_summary.drop(index='Jumlah Gempa', errors='ignore')
            
            # Reindex with all dates and fill missing with 0
            depth_summary_without_total.index = pd.to_datetime(depth_summary_without_total.index)
            depth_summary_without_total.index = depth_summary_without_total.index.strftime('%d-%m-%Y')

            # Plot setup
            fig, ax = plt.subplots(figsize=(12, 6))
            bar_width = 0.25
            x = np.arange(len(depth_summary_without_total.index))
            
            # Plot each category
            for i, category in enumerate(categories_depth):
                ax.bar(x + i * bar_width, 
                    depth_summary_without_total[category], 
                    bar_width, 
                    label=category, 
                    color=colors_depth[i], 
                    edgecolor='black')

            # Chart formatting
            ax.set_title('Frekuensi Gempa Berdasarkan Kedalaman' ,fontsize=14, pad=20)
            ax.set_xlabel('Tanggal', fontsize=12)
            ax.set_ylabel('Jumlah Kejadian', fontsize=12)
            
            # Set x-ticks to show all dates
            ax.set_xticks(x + bar_width)
            
            # Update x-tick labels dengan format baru
            ax.set_xticklabels(depth_summary_without_total.index, rotation=45, ha='right', fontsize=8)
            
            ax.legend(title='Kategori Kedalaman', fontsize=10)
            ax.grid(axis='y', linestyle='--', alpha=0.7)

            for p in ax.patches:
                height = p.get_height()
                if height > 0:  # Hanya tampilkan label jika nilai > 0
                    ax.annotate(f'{int(height)}', 
                            (p.get_x() + p.get_width() / 2, height),
                            ha='center', va='bottom', fontsize=8)

            plt.tight_layout()
            plt.savefig(os.path.join(nama_folder, 'diagbat_depth.png'), dpi=300)
            plt.close()

            worksheet_depth = writer.sheets['Depth Summary']
            worksheet_depth.insert_image('A10', os.path.join(nama_folder, 'diagbat_depth.png'), 
                                    {'x_scale': 1.2, 'y_scale': 1.4})

            # buat digram lingkaran depth
            depth_total = depth_summary.loc['Jumlah Gempa', ['D≤60 km', '60<D≤300 km', 'D>300 km']]
            labels = ['D≤60 km', '60<D≤300 km', 'D>300 km']
            colors = ['red', 'yellow', 'green']
            
            # Membuat figure dan pie chart
            fig, ax = plt.subplots(figsize=(6, 6))
            
            # Fungsi untuk menampilkan persentase dan jumlah kejadian di setiap segmen
            def func(pct, allvals):
                absolute = int(np.round(pct/100.*np.sum(allvals)))
                if absolute > 0:
                    return f'{pct:.1f}%\n({absolute} kejadian)'
                else:
                    return ''  # Jika tidak ada kejadian, label tidak ditampilkan
                    
            # Membuat pie chart
            wedges, texts, autotexts = ax.pie(
                depth_total,
                labels=labels,
                colors=colors,
                startangle=85,
                autopct=lambda pct: func(pct, magnitude_total),  # Menampilkan persentase dan jumlah kejadian
                pctdistance=0.975,  # Jarak persentase dari pusat pie
                labeldistance=9  # Jarak label dari pusat pie (aslinya 1.15)
            )

            # Mengatur ukuran font dan latar belakang putih untuk label dan persentase
            for text in texts:
                text.set_fontsize(12)
                text.set_bbox(dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.2'))

            for autotext in autotexts:
                autotext.set_fontsize(12)
                autotext.set_bbox(dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.2'))

            plt.title('Distribusi Kedalaman', fontsize=11)
            ax.legend(title='Kategori Kedalaman', fontsize=7, title_fontsize='9', loc='lower right')
            plt.tight_layout()
            plt.savefig(os.path.join(nama_folder, 'diaglingk_depth.png'))

            # Insert the magnitude pie chart into Excel
            worksheet_depth.insert_image('W10', os.path.join(nama_folder, 'diaglingk_depth.png'), {'x_scale': 1.2, 'y_scale': 1.2})   
        
        def autofit_columns(file_path, sheet_name):
            """
            Menyesuaikan lebar kolom di Excel berdasarkan panjang konten.
            """
            workbook = load_workbook(file_path)  # Membuka workbook yang telah disimpan
            worksheet = workbook[sheet_name]  # Mendapatkan worksheet berdasarkan nama sheet

            for col in worksheet.columns:
                max_length = 0
                column = col[0].column_letter  # Mendapatkan huruf kolom, misalnya 'A', 'B', dst.
                for cell in col:
                    try:
                        # Mencari panjang maksimum dari setiap kolom
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = (max_length + 2)  # Menambah padding
                worksheet.column_dimensions[column].width = adjusted_width  # Mengatur lebar kolom

            workbook.save(file_path)  # Menyimpan workbook setelah penyesuaian
            workbook.close()  # Menutup workbook untuk memastikan semua perubahan disimpan
        
        # Panggil fungsi utama dengan tanggal yang dipilih
        read_csv_and_analyze(tanggal_awal, tanggal_akhir)

    def print_map(self):
        fault="D:/Seismioto/Bahan/fault.gmt"
        trench="D:/Seismioto/Bahan/trench.gmt"
        legenda="D:/Seismioto/Bahan/legenda.gmt"
        #membuat file .dat
        df = pd.read_csv('datalengkap.csv')
        
        # Memilih kolom yang diinginkan
        selected_columns = df[['Bujur','Lintang','Kedalaman', 'Magnitude']]

        # Menulis data ke file DAT
        with open('inputpeta.dat', 'w') as file:
            for index, row in selected_columns.iterrows():
                line = '  '.join(row.astype(str))  # Menggabungkan semua elemen baris menjadi satu string dengan spasi sebagai pemisah
                file.write(line + '\n')
                
        # Membaca data dari hasil2.csv
        data = pd.read_csv('datalengkap.csv')

        # Buat figure
        fig = pygmt.Figure()

        # Buat color palette untuk kedalaman gempa
        pygmt.makecpt(cmap="geo", series=[-7000, 4000])
        pygmt.makecpt(cmap="red,yellow,green", series="0,60,300,1000", output="quakes.cpt")
        # start_date_formatted = start_date_entry.get().replace('/', '-')  # Ganti format tanggal
        # end_date_formatted = end_date_entry.get().replace('/', '-')
        #title_text = f"Peta Seismisitas Wilayah Bali dan Sekitarnya \nPeriode {start_date_formatted} hingga {end_date_formatted}"
        data['Magnitude'] = pd.to_numeric(data['Magnitude'], errors='coerce')  # Ganti nilai tidak valid dengan NaN

        # Plot peta dasar dengan topografi
        fig.grdimage(
            grid= file_grid,
            region=[113.21, 117.31, -12, -7],
            projection="M17c",
            shading=True,
            frame=["xa2g2", "ya2g2"])#'+t"Peta Seismisitas"'])
        
        # Plot fault lines
        
        fig.plot(data=fault, pen="0.6,black")
        fig.plot(data=trench, pen="4,black")

        fig.plot(x=[114.8, 115.4], y=[-11.5, -7.5], projection="M", pen=2)
        fig.text(x=114.9, y=-11.6, text="A", font="18,Helvetica")
        fig.text(x=115.55, y=-7.5, text="A1", font="18,Helvetica")

        # Plot gempa yang tidak dirasakan (kolom 'Dirasakan' kosong)
        gempa_tidak_dirasakan = data[data['Dirasakan'].isna()]
        if not gempa_tidak_dirasakan.empty:
            fig.plot(
                x=gempa_tidak_dirasakan['Bujur'],
                y=gempa_tidak_dirasakan['Lintang'],
                size=0.08*gempa_tidak_dirasakan['Magnitude'],
                style="cc",  # Lingkaran
                fill=gempa_tidak_dirasakan['Kedalaman'].astype(float),  # Warna berdasarkan kedalaman
                cmap="quakes.cpt",  # Gunakan colormap yang telah ditentukan
                pen="black"
            )
        else:
            print("ada gempa bumi yang dirasakan untuk diplot.")

        # Plot gempa yang dirasakan (kolom 'Dirasakan' terisi)
        gempa_dirasakan = data[~data['Dirasakan'].isna()]
        if not gempa_dirasakan.empty:
            fig.plot(
                x=gempa_dirasakan['Bujur'],
                y=gempa_dirasakan['Lintang'],
                size=0.08*gempa_dirasakan['Magnitude'],
                style="a0.5c",  # Simbol bintang
                fill="red",  # Warna bintang merah
                pen="1p,black"  # Garis tepi hitam dengan ketebalan 1 point
            )
        else:
            print("tidak ada gempa bumi yang dirasakan untuk diplot.")
        
        # Menambahkan colorbar
        fig.image(logobmkg, position='n1.1/0.85+w1.1i')
        fig.image(matangin, position='n1.04/0.7+w1.1i')
        fig.basemap(map_scale="n1.12/0.65+w80k+f0.1p+lKm")
        fig.colorbar(
            frame=["x+lKetinggian", "y"],  # Label untuk sumbu x dan y
            position="n1.24/0.63+w4c/0.5c")  # Posisi colorbar (JMR = tengah-kanan), lebar 0.5 cm, panjang 10 cm
        fig.legend(spec=legenda, position="n1.03/0.04+w5c",box="+gwhite+p1p,black")
        with fig.inset(position="n1.03/0.28+w5c/2.5c", box="+pblack"):
            fig.coast(
                region=[97, 140, -15, 7],
                shorelines='0.5p,black',
                projection="M5c",
                land="green",
                water="lightblue"
            )
            fig.plot(x=[117.31, 113.21, 113.21, 117.31, 117.31], y=[-7, -7, -12, -12, -7], pen="1p,red")

        # Membuat cross section
        pygmt.project(
            data="inputpeta.dat",
            unit=True,
            center=[114.8, -11.5],
            endpoint=[115.4, -7.5],
            width=[-200, 200],
            convention="pz",
            outfile="crsx.dat"
        )

        fig.shift_origin(yshift=9, xshift=17.5)
        fig.plot(
            x=[113, 113, 114.18, 114.18],  # Koordinat X dari sudut-sudut persegi
            y=[-11.4, -12, -12, -11.4],  # Koordinat Y dari sudut-sudut persegi
            fill="white",  # Warna latar belakang
            close=True  # Menutup jalur untuk membuat persegi berisi warna 
        )

        # Buat peta cross section dengan basemap
        fig.basemap(
            projection="X4c/-2.5c",
            region=[0, 450, 0, 350],
            frame=['xafg+lDistance (km)','yafg+lDepth (km)', "wsEN"]
        )

        # Membuat garis cross section   
        cross = pd.read_csv('crsx.dat', delimiter='\s+', header=None)
        # Define your column names
        header = ['Distance', 'Dept', 'Mag']
        
        # Assign the header
        cross.columns = header

        fig.plot(x=cross.Distance, 
                y=cross.Dept, 
                projection="X4/-2.5", 
                size =0.035 * (2 * cross.Mag), style="cc", 
                fill = cross.Dept, 
                cmap = "quakes.cpt", 
                pen = 'black'),
                #region = region_cs)
        fig.text(x=20, y=25, text="A", font="11,Helvetica")
        fig.text(x=370, y=25, text="A1", font="11,Helvetica")

        # Tentukan nama folder berdasarkan tanggal awal dan akhir
        if tanggal_awal.month == tanggal_akhir.month:
            nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
        else:
            nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

        output_map_filename = os.path.join(nama_folder, f'Peta Seismisitas Periode {nama_folder}.png')
        # Menyimpan gambar
        fig.savefig(fname=output_map_filename)
        #fig.show()
        
        messagebox.showinfo("Cetak Peta", "Peta Berhasil di Cetak")

    def print_report(self):
         # Set locale to Indonesian for date formatting
        locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

        # Fungsi untuk memilih file
        def select_file():
            root = tk.Tk()
            root.withdraw()  # Menyembunyikan jendela utama
            file_path = filedialog.askopenfilename(title="Pilih file laporan_gempa.xlsx", filetypes=[("Excel files", "*.xlsx")])
            return file_path

        # Memilih file Excel
        file_path = select_file()
        if not file_path:
            print("Tidak ada file yang dipilih.")
        else:
            # Load the Excel file
            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])
            
            # Tentukan nama folder berdasarkan tanggal awal dan akhir
            if tanggal_awal.month == tanggal_akhir.month:
                nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
            else:
                nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

            # Ambil tanggal awal dan akhir
            start_date = tanggal_awal
            end_date = tanggal_akhir
    
            # Path to images
            logo_path = "D:/Seismioto/Bahan/LogoBMKG.png"  # Ganti dengan path logo BMKG
            magnitude_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_mag.png"
            magnitude_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_mag.png"
            depth_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_depth.png"
            depth_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_depth.png"
            seismic_map_path = f"D:/Seismioto/Pengolahan/{nama_folder}/Peta Seismisitas Periode {nama_folder}.png"
        
            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

            # Calculate the required values
            total_earthquakes = len(earthquake_data)
            min_magnitude = earthquake_data['Magnitude'].min()
            max_magnitude = earthquake_data['Magnitude'].max()

            # Format tanggal dengan nama bulan dalam bahasa Indonesia
            start_date_formatted = start_date.strftime('%d %B %Y')  # contoh format: 01 Januari 2024
            end_date_formatted = end_date.strftime('%d %B %Y')      # contoh format: 05 September 2024
            #date_formatted=date1.strftime('%d %B %Y')
            #print(date_formatted)

            # Menentukan tanggal awal bulan untuk memulai perhitungan minggu
            start_of_month = start_date.replace(day=1)

            # Hitung nomor minggu relatif terhadap awal bulan
            week_number = (start_date - start_of_month).days // 7 + 1

            # Format bulan dalam bahasa Indonesia
            bulan_formatted = start_date.strftime('%B')

            # Menghitung jumlah kejadian gempabumi total
            total_earthquakes = len(earthquake_data)

            # Menghitung jumlah gempa dengan lat > -9 dan kedalaman < 60 km
            shallow_earthquake_count = earthquake_data[(earthquake_data['Lintang'] > -9) & 
                                                    (earthquake_data['Kedalaman'] < 60)].shape[0]

            # Menghitung jumlah gempa di selatan Pulau Jawa, Bali, dan Lombok
            southern_earthquake_count = total_earthquakes - shallow_earthquake_count

            # Magnitude categories
            magnitude_below_3 = earthquake_data[earthquake_data['Magnitude'] <= 3]
            magnitude_3_to_5 = earthquake_data[(earthquake_data['Magnitude'] > 3) & (earthquake_data['Magnitude'] <= 5)]
            magnitude_above_5 = earthquake_data[earthquake_data['Magnitude'] > 5]

            count_below_3 = len(magnitude_below_3)
            count_3_to_5 = len(magnitude_3_to_5)
            count_above_5 = len(magnitude_above_5)

            percentage_below_3 = int((count_below_3 / total_earthquakes) * 100)
            percentage_3_to_5 = int((count_3_to_5 / total_earthquakes) * 100)
            percentage_above_5 = int((count_above_5 / total_earthquakes) * 100)

            # Magnitude categories
            depth_below_60 = earthquake_data[earthquake_data['Kedalaman'] < 60]
            depth_60_to_300 = earthquake_data[(earthquake_data['Kedalaman'] >= 60) & (earthquake_data['Kedalaman'] <= 300)]
            depth_above_300 = earthquake_data[earthquake_data['Kedalaman'] > 300]

            count_depth_below_60 = len(depth_below_60)
            count_depth_60_to_300 = len(depth_60_to_300)
            count_depth_above_300 = len(depth_above_300)

            percentage_depth_below_60 = int((count_depth_below_60 / total_earthquakes) * 100)
            percentage_depth_60_to_300 = int((count_depth_60_to_300 / total_earthquakes) * 100)
            percentage_depth_above_300 = int((count_depth_above_300 / total_earthquakes) * 100)

            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Create a new Word document
            document = Document()

            # Ukuran kertas F4 (21.0 cm x 33.0 cm)
            f4_width = Cm(21.0)
            f4_height = Cm(33.0)

            # Mengatur ukuran kertas F4 untuk setiap section di dokumen
            for section in document.sections:
                # Mengatur orientasi potrait untuk F4
                section.orientation = WD_ORIENT.PORTRAIT  # Bisa diubah ke WD_ORIENT.LANDSCAPE jika ingin landscape
                section.page_width = f4_width
                section.page_height = f4_height

                # Mengatur margin halaman (optional, bisa disesuaikan)
                section.top_margin = Cm(2.5)     # Marginn atas
                section.bottom_margin = Cm(2.5)  # Marginn bawah
                section.left_margin = Cm(2.5)    # Marginn kiri
                section.right_margin = Cm(2.5)   # Marginn kanan

            # Add Logo at the top
            # logo_paragraph = document.add_paragraph()
            # logo_run = logo_paragraph.add_run()
            # logo_run.add_picture(logo_path, width=Inches(1))  # Menambahkan logo dengan lebar 1.5 inches
            # logo_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # # Add Title
            title = document.add_paragraph()
            title_run = title.add_run(f'STASIUN GEOFISIKA DENPASAR                                                                                              LAPORAN AKTIVITAS SEISMISITAS WILAYAH BALI DAN SEKITARNYA                                                                                              PERIODE {start_date_formatted} – {end_date_formatted}')
            title_run.bold = True
            title_run.font.size = Pt(14)
            title_run.font.all_caps = True  # Membuat semua huruf kapital
            title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add Subtitle
            # subtitle = document.add_paragraph()
            # subtitle_run = subtitle.add_run(f'STASIUN GEOFISIKA DENPASAR                                                                                              LAPORAN AKTIVITAS SEISMISITAS WILAYAH BALI DAN SEKITARNYA                                                                                              PERIODE {start_date_formatted} – {end_date_formatted}')
            # subtitle_run.font.size = Pt(12)
            # subtitle_run.font.all_caps = True  # Membuat semua huruf kapital
            # subtitle.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # # Add Date Period
            # period = document.add_paragraph()
            # period_run = period.add_run(f'PERIODE {start_date_formatted} – {end_date_formatted}')
            # period_run.font.size = Pt(12)
            # period_run.font.all_caps = True  # Membuat semua huruf kapital
            # period.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')

            # Teks Deskriptif Magnitudo dengan perhitungan minggu bulanan
            magnitude_descriptive_text = (
                f"Berdasarkan data Stasiun Geofisika Denpasar selama minggu ke-{week_number} bulan "
                f"{bulan_formatted} {start_date.year}, "
                f"di daerah Bali dan sekitarnya telah terjadi {total_earthquakes} kejadian gempabumi dengan magnitudo bervariasi "
                f"mulai dari M {min_magnitude} sampai M {max_magnitude}. Kejadian gempabumi dengan magnitudo M<3 sejumlah "
                f"{count_below_3} kejadian atau {percentage_below_3}% dari total kejadian gempabumi. Sedangkan untuk M 3 – 5 "
                f"sejumlah {count_3_to_5} kejadian atau {percentage_3_to_5}% dari total kejadian gempabumi "
            )

            # Conditional text for magnitude above 5
            if count_above_5 > 0:
                magnitude_descriptive_text += (
                    f"dan kejadian gempa M>5 sejumlah {count_above_5} kejadian atau {percentage_above_5}% dari total kejadian gempabumi."
                )
            else:
                magnitude_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan magnitudo M>5."

            # Menambahkan paragraf dengan teks deskriptif magnitudo
            magnitude_paragraph = document.add_paragraph(magnitude_descriptive_text)

            # Mengatur alignment paragraf menjadi justify
            magnitude_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Add Magnitude Bar Chart Image
            try:
                document.add_paragraph().add_run().add_picture(magnitude_bar_chart_path, width=Inches(6.56))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM BATANG MAGNITUDO TIDAK DITEMUKAN').bold = True

            # Add Magnitude Pie Chart Image
            try:
                document.add_paragraph().add_run().add_picture(magnitude_pie_chart_path, width=Inches(3.15))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM LINGKARAN MAGNITUDO TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Magnitudo
            document.add_paragraph('Gambar 1. Histogram gempabumi harian berdasarkan magnitudo (atas) dan diagram gempabumi berdasarkan magnitudo (bawah)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Add space
            document.add_paragraph('')

            # Teks Deskriptif Kedalaman
            depth_descriptive_text = (
                f"Berdasarkan kedalaman, gempabumi yang mendominasi adalah kejadian gempabumi {dominant_category}  dari total kejadian gempabumi. "
                f"Jumlah gempabumi dangkal sebanyak {count_depth_below_60} kejadian gempabumi, "
                f"terdapat sebanyak {count_depth_60_to_300} kejadian gempabumi dengan kedalaman menengah 60 km - 300 km atau {percentage_depth_60_to_300}% dari total kejadian gempabumi, "
            )

            # Conditional text for depth above 300 km
            if count_depth_above_300 > 0:
                depth_descriptive_text += (
                    f"dan kejadian gempa dalam sejumlah {count_depth_above_300} kejadian atau {percentage_depth_above_300}% dari total kejadian gempabumi."
                )
            else:
                depth_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan kedalaman dalam >300 km."

            depth_paragraph = document.add_paragraph(depth_descriptive_text)
            depth_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Add Depth Bar Chart Image
            try:
                document.add_paragraph().add_run().add_picture(depth_bar_chart_path, width=Inches(6.56))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM BATANG KEDALAMAN TIDAK DITEMUKAN').bold = True

            # Add Depth Pie Chart Image
            try:
                document.add_paragraph().add_run().add_picture(depth_pie_chart_path, width=Inches(3.15))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM LINGKARAN KEDALAMAN TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Kedalaman (akan diisi sesuai format)
            document.add_paragraph('Gambar 2.  Histogram gempabumi harian berdasarkan kedalaman (atas) dan diagram gempabumi berdasarkan kedalaman (bawah)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add Seismic Map Image
            try:
                document.add_paragraph().add_run().add_picture(seismic_map_path, width=Inches(6))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('PETA SEISMISITAS TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Peta Seismisitas
            peta_seismisitas_text = (
                f"Gambar 3. Peta seismisitas Bali dan sekitarnya periode {start_date_formatted} - {end_date_formatted}"
            )
            peta_seismisitas_paragraph = document.add_paragraph(peta_seismisitas_text)
            peta_seismisitas_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')

            # Teks analisis yang diminta
            analisis_text1 = (
                f"Selama satu minggu terakhir, gempabumi yang terjadi di Bali dan sekitarnya dikelompokan berdasarkan sumbernya (Gambar 3):\n"
            )    

            # Menambahkan paragraf analisis ke dalam dokumen
            analisis_paragraph1 = document.add_paragraph(analisis_text1)
            analisis_paragraph1.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

            analisis_text2 = (
                f"1. Sebanyak {southern_earthquake_count} kejadian gempabumi terjadi di selatan Pulau Jawa, Bali, dan Lombok "
                f"yang diperkirakan berasosiasi dengan subduksi Indo Australia-Eurasia.\n"
                f"2. Sebanyak {shallow_earthquake_count} kejadian gempabumi terjadi menyebar di wilayah Pulau Bali, Lombok dan sekitarnya "
                f"diperkirakan berkaitan dengan sesar di belakang busur Flores atau Flores Back Arc Thrust dan aktivitas sesar aktif."
            )

            # Menambahkan paragraf analisis ke dalam dokumen
            analisis_paragraph2 = document.add_paragraph(analisis_text2)
            analisis_paragraph2.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

            # Menghitung jumlah kejadian gempa yang dirasakan
            earthquakes_felt = earthquake_data[earthquake_data['Dirasakan'].notna()]
            total_felt = len(earthquakes_felt)

            # Menyusun narasi pembuka
            if total_felt > 0:
                felt_intro = (
                    f"Selama periode ini terdapat {total_felt} kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )
            else:
                felt_intro = (
                    f"Selama periode ini tidak terdapat kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )

            # Menambahkan narasi pembuka ke dokumen
            felt_intro_paragraph = document.add_paragraph(felt_intro)
            felt_intro_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Menyusun detail setiap kejadian gempa yang dirasakan dalam format list dengan nomor urut
            if total_felt > 0:
                # Inisialisasi counter untuk nomor urut
                counter = 1

                for index, row in earthquakes_felt.iterrows():
                    magnitude = row['Magnitude']
                    date = row['Tanggal'].strftime('%d %B %Y')  # Format tanggal DD/MM/YYYY
                    time = row['Waktu (WIB)']  # Asumsikan kolom waktu sudah dalam format string yang benar
                    location = row['Dirasakan']
                    description = row['Keterangan']
                    depth = row['Kedalaman']

                    # Format detail gempa yang dirasakan dengan nomor urut
                    felt_detail = (
                        f"{counter}. Gempabumi dengan magnitudo {magnitude} {location} yang terjadi pada tanggal {date} "
                        f"pukul {time} WIB berlokasi di {description} pada kedalaman {depth} km."
                    )

                    # Menambahkan detail gempa ke dalam dokumen sebagai paragraf terpisah
                    felt_detail_paragraph = document.add_paragraph(felt_detail)
                    felt_detail_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

                    # Meningkatkan counter untuk nomor urut berikutnya
                    counter += 1
            else:
                # Jika tidak ada gempa yang dirasakan, tambahkan keterangan tambahan
                no_felt_detail = document.add_paragraph(" ")
                no_felt_detail.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER


            # Tambahkan section baru dengan orientasi landscape
            new_section = document.add_section(WD_SECTION.NEW_PAGE)
            new_section.orientation = WD_ORIENT.LANDSCAPE

            # Atur ukuran kertas agar landscape sesuai
            new_section.page_width, new_section.page_height = new_section.page_height, new_section.page_width

            # Menentukan margin untuk section baru agar tabel berada di tengah secara vertikal
            section_margin = Cm(2)  # Sesuaikan margin jika perlu
            new_section.top_margin = section_margin
            new_section.bottom_margin = section_margin
            new_section.left_margin = section_margin
            new_section.right_margin = section_margin

            # Menambahkan paragraf baru untuk membungkus tabel
            table_paragraph = document.add_paragraph()
            table_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment paragraf ke tengah

            # Membuat teks keterangan tabel gempabumi
            tabel_keterangan_text = (
                f"Tabel 1. Data gempabumi di Bali dan sekitarnya tanggal {start_date_formatted} – {end_date_formatted}"
            )

            # Menambahkan teks keterangan tabel ke dalam dokumen
            tabel_keterangan_paragraph = document.add_paragraph(tabel_keterangan_text)
            tabel_keterangan_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment menjadi tengah

            # Menambahkan tabel langsung ke dalam paragraf
            table = document.add_table(rows=1, cols=len(earthquake_data.columns))
            table.style = 'Table Grid'

            # Set table alignment to center
            table.alignment = WD_TABLE_ALIGNMENT.CENTER

            # Set column widths to match the text content
            column_widths = {
                "NO.": 1.5,  # NO
                "Tanggal": 2.5,  # TANGGAL
                "Waktu (WIB)": 2.5,  # WAKTU (WIB)
                "Lintang": 2.0,  # LATITUDE
                "Bujur": 2.0,  # LONGITUDE
                "Kedalaman": 2.0,  # Depth
                "Magnitude": 1.5,  # Magnitude
                "Keterangan": 5.0,  # Keterangan
                "Dirasakan": 2.5  # Dirasakan
            }

            # Add header row
            hdr_cells = table.rows[0].cells
            for i, column_name in enumerate(earthquake_data.columns):
                hdr_cells[i].text = column_name
                hdr_cells[i].width = Cm(column_widths.get(column_name, 2.0))  # Set width based on column

            # Add data rows, memformat kolom 'Tanggal' tanpa jam
            for index, row in earthquake_data.iterrows():
                row_cells = table.add_row().cells
                for i, value in enumerate(row):
                    # Jika kolom adalah 'Tanggal', hanya tampilkan tanggal tanpa jam
                    if earthquake_data.columns[i] == 'Tanggal':
                        row_cells[i].text = value.strftime('%d/%m/%Y')  # Format tanggal menjadi DD/MM/YYYY
                    # Jika kolom adalah 'Dirasakan' dan nilai adalah NaN, ganti dengan teks yang sesuai
                    elif earthquake_data.columns[i] == 'Dirasakan':
                        # Ganti NaN dengan string kosong atau teks lain
                        row_cells[i].text = '' if pd.isna(value) else str(value)
                    else:
                        row_cells[i].text = str(value)

                    # Set the width of each data cell, sesuai dengan kolom
                    row_cells[i].width = Cm(column_widths.get(earthquake_data.columns[i], 2.0))

            # Mengatur tabel agar berada di tengah secara horizontal
            table.alignment = WD_TABLE_ALIGNMENT.CENTER
            
            
            # Simpan 'datalengkap.csv' ke folder baru dengan nama 'data gempa (nama folder).xlsx'
            output_file = os.path.join(nama_folder, f'Laporan Gempabumi Periode {nama_folder}.docx')
        
            # Save the document
            document.save(output_file)

        messagebox.showinfo("Cetak Laporan", "Berhasil Mencetak laporan ke file Word")

    def print_press_release(self):
        # Set locale to Indonesian for date formatting
        locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

        # Fungsi untuk memilih file
        def select_file():
            root = tk.Tk()
            root.withdraw()  # Menyembunyikan jendela utama
            file_path = filedialog.askopenfilename(title="Pilih file laporan_gempa.xlsx", filetypes=[("Excel files", "*.xlsx")])
            return file_path

        # Memilih file Excel
        file_path = select_file()
        if not file_path:
            print("Tidak ada file yang dipilih.")
        else:
            # Load the Excel file
            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

            # Tentukan nama folder berdasarkan tanggal awal dan akhir
            if tanggal_awal.month == tanggal_akhir.month:
                nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
            else:
                nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

            # Ambil tanggal awal dan akhir
            start_date = tanggal_awal
            end_date = tanggal_akhir
    
            # Path to images
            magnitude_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_mag.png"
            magnitude_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_mag.png"
            depth_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_depth.png"
            depth_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_depth.png"
            seismic_map_path = f"D:/Seismioto/Pengolahan/{nama_folder}/Peta Seismisitas Periode {nama_folder}.png"
        
            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

            # Calculate the required values
            total_earthquakes = len(earthquake_data)
            min_magnitude = earthquake_data['Magnitude'].min()
            max_magnitude = earthquake_data['Magnitude'].max()

            # Format tanggal dengan nama bulan dalam bahasa Indonesia
            start_date_formatted = start_date.strftime('%d %B %Y')  # contoh format: 01 Januari 2024
            end_date_formatted = end_date.strftime('%d %B %Y')      # contoh format: 05 September 2024

            # Menentukan tanggal awal bulan untuk memulai perhitungan minggu
            start_of_month = start_date.replace(day=1)

            # Hitung nomor minggu relatif terhadap awal bulan
            week_number = (start_date - start_of_month).days // 7 + 1

            # Format bulan dalam bahasa Indonesia
            bulan_formatted = start_date.strftime('%B')

            # Menghitung jumlah kejadian gempabumi total
            total_earthquakes = len(earthquake_data)

            # Menghitung jumlah gempa dengan lat > -9 dan kedalaman < 60 km
            shallow_earthquake_count = earthquake_data[(earthquake_data['Lintang'] > -9) & 
                                                    (earthquake_data['Kedalaman'] < 60)].shape[0]

            # Menghitung jumlah gempa di selatan Pulau Jawa, Bali, dan Lombok
            southern_earthquake_count = total_earthquakes - shallow_earthquake_count

            # Magnitude categories
            magnitude_below_3 = earthquake_data[earthquake_data['Magnitude'] <= 3]
            magnitude_3_to_5 = earthquake_data[(earthquake_data['Magnitude'] > 3) & (earthquake_data['Magnitude'] <= 5)]
            magnitude_above_5 = earthquake_data[earthquake_data['Magnitude'] > 5]

            count_below_3 = len(magnitude_below_3)
            count_3_to_5 = len(magnitude_3_to_5)
            count_above_5 = len(magnitude_above_5)

            percentage_below_3 = int((count_below_3 / total_earthquakes) * 100)
            percentage_3_to_5 = int((count_3_to_5 / total_earthquakes) * 100)
            percentage_above_5 = int((count_above_5 / total_earthquakes) * 100)

            # Magnitude categories
            depth_below_60 = earthquake_data[earthquake_data['Kedalaman'] < 60]
            depth_60_to_300 = earthquake_data[(earthquake_data['Kedalaman'] >= 60) & (earthquake_data['Kedalaman'] <= 300)]
            depth_above_300 = earthquake_data[earthquake_data['Kedalaman'] > 300]

            count_depth_below_60 = len(depth_below_60)
            count_depth_60_to_300 = len(depth_60_to_300)
            count_depth_above_300 = len(depth_above_300)

            percentage_depth_below_60 = int((count_depth_below_60 / total_earthquakes) * 100)
            percentage_depth_60_to_300 = int((count_depth_60_to_300 / total_earthquakes) * 100)
            percentage_depth_above_300 = int((count_depth_above_300 / total_earthquakes) * 100)

            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Create a new Word document
            document = Document()

            # Ukuran kertas F4 (21.0 cm x 33.0 cm)
            f4_width = Cm(21.0)
            f4_height = Cm(33.0)

            # Mengatur ukuran kertas F4 untuk setiap section di dokumen
            for section in document.sections:
                # Mengatur orientasi potrait untuk F4
                section.orientation = WD_ORIENT.PORTRAIT  # Bisa diubah ke WD_ORIENT.LANDSCAPE jika ingin landscape
                section.page_width = f4_width
                section.page_height = f4_height

                # Mengatur margin halaman (optional, bisa disesuaikan)
                section.top_margin = Cm(2.5)     # Marginn atas
                section.bottom_margin = Cm(2.5)  # Marginn bawah
                section.left_margin = Cm(2.5)    # Marginn kiri
                section.right_margin = Cm(2.5)   # Marginn kanan

            # Add Title
            title = document.add_paragraph()
            title_run = title.add_run(f'PRESS RELEASE AKTIVITAS SEISMISITAS                                                                                              WILAYAH BALI DAN SEKITARNYA                                                                                                        PERIODE {start_date_formatted} – {end_date_formatted}')
            title_run.bold = True
            title_run.font.size = Pt(14)
            title_run.font.all_caps = True  # Membuat semua huruf kapital
            title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')

            # Teks Deskriptif Magnitudo dengan perhitungan minggu bulanan
            magnitude_descriptive_text = (
                f"Berdasarkan data Stasiun Geofisika Denpasar selama minggu ke-{week_number} bulan "
                f"{bulan_formatted} {start_date.year}, "
                f"di daerah Bali dan sekitarnya telah terjadi {total_earthquakes} kejadian gempabumi dengan magnitudo bervariasi "
                f"mulai dari M {min_magnitude} sampai M {max_magnitude}. Kejadian gempabumi dengan magnitudo M<3 sejumlah "
                f"{count_below_3} kejadian atau {percentage_below_3}% dari total kejadian gempabumi. Sedangkan untuk M 3 – 5 "
                f"sejumlah {count_3_to_5} kejadian atau {percentage_3_to_5}% dari total kejadian gempabumi "
            )

            # Conditional text for magnitude above 5
            if count_above_5 > 0:
                magnitude_descriptive_text += (
                    f"dan kejadian gempa M>5 sejumlah {count_above_5} kejadian atau {percentage_above_5}% dari total kejadian gempabumi."
                )
            else:
                magnitude_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan magnitudo M>5."

            # Menambahkan paragraf dengan teks deskriptif magnitudo
            magnitude_paragraph = document.add_paragraph(magnitude_descriptive_text)

            # Mengatur alignment paragraf menjadi justify
            magnitude_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            
            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Add space
            document.add_paragraph('')

            # Teks Deskriptif Kedalaman
            depth_descriptive_text = (
                f"Berdasarkan kedalaman, gempabumi yang mendominasi adalah kejadian gempabumi {dominant_category}  dari total kejadian gempabumi. "
                f"Jumlah gempabumi dangkal sebanyak {count_depth_below_60} kejadian gempabumi, "
                f"terdapat sebanyak {count_depth_60_to_300} kejadian gempabumi dengan kedalaman menengah 60 km - 300 km atau {percentage_depth_60_to_300}% dari total kejadian gempabumi, "
            )

            # Conditional text for depth above 300 km
            if count_depth_above_300 > 0:
                depth_descriptive_text += (
                    f"dan kejadian gempa dalam sejumlah {count_depth_above_300} kejadian atau {percentage_depth_above_300}% dari total kejadian gempabumi."
                )
            else:
                depth_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan kedalaman dalam >300 km."

            depth_paragraph = document.add_paragraph(depth_descriptive_text)
            depth_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY   

            # Add space
            document.add_paragraph('')

            # Teks analisis yang diminta
            analisis_text1 = (
                f"Selama satu minggu terakhir, gempabumi yang terjadi di Bali dan sekitarnya dikelompokan berdasarkan sumbernya (Gambar 3):\n"
            )    

            # Menambahkan paragraf analisis ke dalam dokumen
            analisis_paragraph1 = document.add_paragraph(analisis_text1)
            analisis_paragraph1.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

            analisis_text2 = (
                f"1. Sebanyak {southern_earthquake_count} kejadian gempabumi terjadi di selatan Pulau Jawa, Bali, dan Lombok "
                f"yang diperkirakan berasosiasi dengan subduksi Indo Australia-Eurasia.\n"
                f"2. Sebanyak {shallow_earthquake_count} kejadian gempabumi terjadi menyebar di wilayah Pulau Bali, Lombok dan sekitarnya "
                f"diperkirakan berkaitan dengan sesar di belakang busur Flores atau Flores Back Arc Thrust dan aktivitas sesar aktif."
            )

            # Menambahkan paragraf analisis ke dalam dokumen
            analisis_paragraph2 = document.add_paragraph(analisis_text2)
            analisis_paragraph2.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

            # Menghitung jumlah kejadian gempa yang dirasakan
            earthquakes_felt = earthquake_data[earthquake_data['Dirasakan'].notna()]
            total_felt = len(earthquakes_felt)

            # Menyusun narasi pembuka
            if total_felt > 0:
                felt_intro = (
                    f"Selama periode ini terdapat {total_felt} kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )
            else:
                felt_intro = (
                    f"Selama periode ini tidak terdapat kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )

            # Menambahkan narasi pembuka ke dokumen
            felt_intro_paragraph = document.add_paragraph(felt_intro)
            felt_intro_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY    

            # Menyusun detail setiap kejadian gempa yang dirasakan dalam format list dengan nomor urut
            if total_felt > 0:
                # Inisialisasi counter untuk nomor urut
                counter = 1

                for index, row in earthquakes_felt.iterrows():
                    magnitude = row['Magnitude']
                    date = row['Tanggal'].strftime('%d %B %Y')  # Format tanggal DD/MM/YYYY
                    time = row['Waktu (WIB)']  # Asumsikan kolom waktu sudah dalam format string yang benar
                    location = row['Dirasakan']
                    description = row['Keterangan']
                    depth = row['Kedalaman']

                    # Format detail gempa yang dirasakan dengan nomor urut
                    felt_detail = (
                        f"{counter}. Gempabumi dengan magnitudo {magnitude} {location} yang terjadi pada tanggal {date} "
                        f"pukul {time} WIB berlokasi di {description} pada kedalaman {depth} km."
                    )

                    # Menambahkan detail gempa ke dalam dokumen sebagai paragraf terpisah
                    felt_detail_paragraph = document.add_paragraph(felt_detail)
                    felt_detail_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

                    # Meningkatkan counter untuk nomor urut berikutnya
                    counter += 1
            else:
                # Jika tidak ada gempa yang dirasakan, tambahkan keterangan tambahan
                no_felt_detail = document.add_paragraph(" ")
                no_felt_detail.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add Magnitude Bar Chart Image
            try:
                document.add_paragraph().add_run().add_picture(magnitude_bar_chart_path, width=Inches(5.65),) 
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM BATANG MAGNITUDO TIDAK DITEMUKAN').bold = True

            # Add Magnitude Pie Chart Image
            try:
                document.add_paragraph().add_run().add_picture(magnitude_pie_chart_path, width=Inches(3.15))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM LINGKARAN MAGNITUDO TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Magnitudo
            document.add_paragraph('Gambar 1. Histogram gempabumi harian berdasarkan magnitudo (kiri) dan diagram gempabumi berdasarkan magnitudo (kanan)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')

            # Add Depth Bar Chart Image
            try:
                document.add_paragraph().add_run().add_picture(depth_bar_chart_path, width=Inches(5.65))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM BATANG KEDALAMAN TIDAK DITEMUKAN').bold = True

            # Add Depth Pie Chart Image
            try:
                document.add_paragraph().add_run().add_picture(depth_pie_chart_path, width=Inches(3.15))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('DIAGRAM LINGKARAN KEDALAMAN TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Kedalaman (akan diisi sesuai format)
            document.add_paragraph('Gambar 2.  Histogram gempabumi harian berdasarkan kedalaman (kiri) dan diagram gempabumi berdasarkan kedalaman (kanan)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')
            
            # Add Seismic Map Image
            try:
                document.add_paragraph().add_run().add_picture(seismic_map_path, width=Inches(6))
                document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            except FileNotFoundError:
                document.add_paragraph('PETA SEISMISITAS TIDAK DITEMUKAN').bold = True

            # Teks Keterangan Gambar Peta Seismisitas
            peta_seismisitas_text = (
                f"Gambar 3. Peta seismisitas Bali dan sekitarnya periode {start_date_formatted} - {end_date_formatted}"
            )
            peta_seismisitas_paragraph = document.add_paragraph(peta_seismisitas_text)
            peta_seismisitas_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Tambahkan section baru dengan orientasi landscape
            new_section = document.add_section(WD_SECTION.NEW_PAGE)
            new_section.orientation = WD_ORIENT.LANDSCAPE

            # Atur ukuran kertas agar landscape sesuai
            new_section.page_width, new_section.page_height = new_section.page_height, new_section.page_width

            # Menentukan margin untuk section baru agar tabel berada di tengah secara vertikal
            section_margin = Cm(2)  # Sesuaikan margin jika perlu
            new_section.top_margin = section_margin
            new_section.bottom_margin = section_margin
            new_section.left_margin = section_margin
            new_section.right_margin = section_margin

            # Menambahkan paragraf baru untuk membungkus tabel
            table_paragraph = document.add_paragraph()
            table_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment paragraf ke tengah

            # Membuat teks keterangan tabel gempabumi
            tabel_keterangan_text = (
                f"Tabel 1. Data gempabumi di Bali dan sekitarnya tanggal {start_date_formatted} – {end_date_formatted}"
            )

            # Menambahkan teks keterangan tabel ke dalam dokumen
            tabel_keterangan_paragraph = document.add_paragraph(tabel_keterangan_text)
            tabel_keterangan_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment menjadi tengah

            # Menambahkan tabel langsung ke dalam paragraf
            table = document.add_table(rows=1, cols=len(earthquake_data.columns))
            table.style = 'Table Grid'

            # Set table alignment to center
            table.alignment = WD_TABLE_ALIGNMENT.CENTER

            # Set column widths to match the text content
            column_widths = {
                "NO.": 1.5,  # NO
                "Tanggal": 2.5,  # TANGGAL
                "Waktu (WIB)": 2.5,  # WAKTU (WIB)
                "Lintang": 2.0,  # LATITUDE
                "Bujur": 2.0,  # LONGITUDE
                "Kedalaman": 2.0,  # Depth
                "Magnitude": 1.5,  # Magnitude
                "Keterangan": 5.0,  # Keterangan
                "Dirasakan": 2.5  # Dirasakan
            }

            # Add header row
            hdr_cells = table.rows[0].cells
            for i, column_name in enumerate(earthquake_data.columns):
                hdr_cells[i].text = column_name
                hdr_cells[i].width = Cm(column_widths.get(column_name, 2.0))  # Set width based on column

            # Add data rows, memformat kolom 'Tanggal' tanpa jam
            for index, row in earthquake_data.iterrows():
                row_cells = table.add_row().cells
                for i, value in enumerate(row):
                    # Jika kolom adalah 'Tanggal', hanya tampilkan tanggal tanpa jam
                    if earthquake_data.columns[i] == 'Tanggal':
                        row_cells[i].text = value.strftime('%d/%m/%Y')  # Format tanggal menjadi DD/MM/YYYY
                    # Jika kolom adalah 'Dirasakan' dan nilai adalah NaN, ganti dengan teks yang sesuai
                    elif earthquake_data.columns[i] == 'Dirasakan':
                        # Ganti NaN dengan string kosong atau teks lain
                        row_cells[i].text = '' if pd.isna(value) else str(value)
                    else:
                        row_cells[i].text = str(value)

                    # Set the width of each data cell, sesuai dengan kolom
                    row_cells[i].width = Cm(column_widths.get(earthquake_data.columns[i], 2.0))

            # Mengatur tabel agar berada di tengah secara horizontal
            table.alignment = WD_TABLE_ALIGNMENT.CENTER
            
            
            # Simpan 'datalengkap.csv' ke folder baru dengan nama 'data gempa (nama folder).xlsx'
            output_file = os.path.join(nama_folder, f'Press Release Gempabumi Periode {nama_folder}.docx')
        
            # Save the document
            document.save(output_file)

        messagebox.showinfo("Cetak Laporan", "Berhasil Mencetak Press Release")

    def print_description(self):
        # Set locale to Indonesian for date formatting
        locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

        # Fungsi untuk memilih file
        def select_file():
            root = tk.Tk()
            root.withdraw()  # Menyembunyikan jendela utama
            file_path = filedialog.askopenfilename(title="Pilih file laporan_gempa.xlsx", filetypes=[("Excel files", "*.xlsx")])
            return file_path

        # Memilih file Excel
        file_path = select_file()
        if not file_path:
            print("Tidak ada file yang dipilih.")
        else:
            # Load the Excel file
            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

            # Ambil tanggal awal dan akhir
            start_date = tanggal_awal
            end_date = tanggal_akhir
            
            # Format tanggal untuk digunakan dalam nama folder     

            earthquake_data = pd.read_excel(file_path)

            # Pastikan kolom 'Tanggal' dalam format datetime
            earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

            # Tentukan nama folder berdasarkan tanggal awal dan akhir
            if tanggal_awal.month == tanggal_akhir.month:
                nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
            else:
                nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

            # Calculate the required values
            total_earthquakes = len(earthquake_data)
            min_magnitude = earthquake_data['Magnitude'].min()
            max_magnitude = earthquake_data['Magnitude'].max()

            # Date range
            start_date = earthquake_data['Tanggal'].min()
            end_date = earthquake_data['Tanggal'].max()
            #date1 = earthquake_data['Tanggal']

            # Format tanggal dengan nama bulan dalam bahasa Indonesia
            start_date_formatted = start_date.strftime('%d %B %Y')  # contoh format: 01 Januari 2024
            end_date_formatted = end_date.strftime('%d %B %Y')      # contoh format: 05 September 2024
            #date_formatted=date1.strftime('%d %B %Y')
            #print(date_formatted)

            # Menentukan tanggal awal bulan untuk memulai perhitungan minggu
            start_of_month = start_date.replace(day=1)

            # Hitung nomor minggu relatif terhadap awal bulan
            week_number = (start_date - start_of_month).days // 7 + 1

            # Format bulan dalam bahasa Indonesia
            bulan_formatted = start_date.strftime('%B')

            # Menghitung jumlah kejadian gempabumi total
            total_earthquakes = len(earthquake_data)

            # Menghitung jumlah gempa dengan lat < -9 dan kedalaman < 60 km
            shallow_earthquake_count = earthquake_data[(earthquake_data['Lintang'] < -9) & 
                                                    (earthquake_data['Kedalaman'] < 60)].shape[0]

            # Menghitung jumlah gempa di selatan Pulau Jawa, Bali, dan Lombok
            southern_earthquake_count = total_earthquakes - shallow_earthquake_count

            # Magnitude categories
            magnitude_below_3 = earthquake_data[earthquake_data['Magnitude'] <= 3]
            magnitude_3_to_5 = earthquake_data[(earthquake_data['Magnitude'] > 3) & (earthquake_data['Magnitude'] <= 5)]
            magnitude_above_5 = earthquake_data[earthquake_data['Magnitude'] > 5]

            count_below_3 = len(magnitude_below_3)
            count_3_to_5 = len(magnitude_3_to_5)
            count_above_5 = len(magnitude_above_5)

            percentage_below_3 = int((count_below_3 / total_earthquakes) * 100)
            percentage_3_to_5 = int((count_3_to_5 / total_earthquakes) * 100)
            percentage_above_5 = int((count_above_5 / total_earthquakes) * 100)

            # Magnitude categories
            depth_below_60 = earthquake_data[earthquake_data['Kedalaman'] < 60]
            depth_60_to_300 = earthquake_data[(earthquake_data['Kedalaman'] >= 60) & (earthquake_data['Kedalaman'] <= 300)]
            depth_above_300 = earthquake_data[earthquake_data['Kedalaman'] > 300]

            count_depth_below_60 = len(depth_below_60)
            count_depth_60_to_300 = len(depth_60_to_300)
            count_depth_above_300 = len(depth_above_300)

            percentage_depth_below_60 = int((count_depth_below_60 / total_earthquakes) * 100)
            percentage_depth_60_to_300 = int((count_depth_60_to_300 / total_earthquakes) * 100)
            percentage_depth_above_300 = int((count_depth_above_300 / total_earthquakes) * 100)

            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Create a new Word document
            document = Document()

            # Ukuran kertas F4 (21.0 cm x 33.0 cm)
            f4_width = Cm(21.0)
            f4_height = Cm(33.0)

            # Mengatur ukuran kertas F4 untuk setiap section di dokumen
            for section in document.sections:
                # Mengatur orientasi potrait untuk F4
                section.orientation = WD_ORIENT.PORTRAIT  # Bisa diubah ke WD_ORIENT.LANDSCAPE jika ingin landscape
                section.page_width = f4_width
                section.page_height = f4_height

                # Mengatur margin halaman (optional, bisa disesuaikan)
                section.top_margin = Cm(2.5)     # Marginn atas
                section.bottom_margin = Cm(2.5)  # Marginn bawah
                section.left_margin = Cm(2.5)    # Marginn kiri
                section.right_margin = Cm(2.5)   # Marginn kanan

            # Add Subtitle
            subtitle = document.add_paragraph()
            subtitle_run = subtitle.add_run('Informasi Gempabumi di Wilayah Bali dan Sekitarnya')
            subtitle_run.font.size = Pt(12)
            subtitle.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add Date Period
            period = document.add_paragraph()
            period_run = period.add_run(f'Periode {nama_folder}')
            period_run.font.size = Pt(12)
            period.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')

            # Teks Deskriptif Magnitudo dengan perhitungan minggu bulanan
            magnitude_descriptive_text = (
                f"Berdasarkan data Stasiun Geofisika Denpasar selama minggu ke-{week_number} bulan "
                f"{bulan_formatted} {start_date.year}, "
                f"di daerah Bali dan sekitarnya telah terjadi {total_earthquakes} kejadian gempabumi dengan magnitudo bervariasi "
                f"mulai dari M {min_magnitude} sampai M {max_magnitude}. Kejadian gempabumi dengan magnitudo M<3 sejumlah "
                f"{count_below_3} kejadian atau {percentage_below_3}% dari total kejadian gempabumi. Sedangkan untuk M 3 – 5 "
                f"sejumlah {count_3_to_5} kejadian atau {percentage_3_to_5}% dari total kejadian gempabumi "
            )

            # Conditional text for magnitude above 5
            if count_above_5 > 0:
                magnitude_descriptive_text += (
                    f"dan kejadian gempa M>5 sejumlah {count_above_5} kejadian atau {percentage_above_5}% dari total kejadian gempabumi."
                )
            else:
                magnitude_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan magnitudo M>5."

            # Menambahkan paragraf dengan teks deskriptif magnitudo
            magnitude_paragraph = document.add_paragraph(magnitude_descriptive_text)

            # Mengatur alignment paragraf menjadi justify
            magnitude_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Determine dominant depth category
            dominant_category = ''
            if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
                dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
            elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
                dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
            else:
                dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

            # Teks Deskriptif Kedalaman
            depth_descriptive_text = (
                f"Berdasarkan kedalaman, gempabumi yang mendominasi adalah kejadian gempabumi {dominant_category}  dari total kejadian gempabumi. "
                f"Jumlah gempabumi dangkal sebanyak {count_depth_below_60} kejadian gempabumi, "
                f"terdapat sebanyak {count_depth_60_to_300} kejadian gempabumi dengan kedalaman menengah 60 km - 300 km atau {percentage_depth_60_to_300}% dari total kejadian gempabumi, "
            )

            # Conditional text for depth above 300 km
            if count_depth_above_300 > 0:
                depth_descriptive_text += (
                    f"dan kejadian gempa dalam sejumlah {count_depth_above_300} kejadian atau {percentage_depth_above_300}% dari total kejadian gempabumi."
                )
            else:
                depth_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan kedalaman dalam >300 km."

            depth_paragraph = document.add_paragraph(depth_descriptive_text)
            depth_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Menghitung jumlah kejadian gempa yang dirasakan
            earthquakes_felt = earthquake_data[earthquake_data['Dirasakan'].notna()]
            total_felt = len(earthquakes_felt)

            # Menyusun narasi pembuka
            if total_felt > 0:
                felt_intro = (
                    f"Selama periode ini terdapat {total_felt} kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )
            else:
                felt_intro = (
                    f"Selama periode ini tidak terdapat kejadian gempabumi yang dirasakan oleh masyarakat "
                    f"di wilayah Bali dan sekitarnya."
                )

            # Menambahkan narasi pembuka ke dokumen
            felt_intro_paragraph = document.add_paragraph(felt_intro)
            felt_intro_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

            # Menyusun detail setiap kejadian gempa yang dirasakan dalam format list dengan nomor urut
            if total_felt > 0:
                # Inisialisasi counter untuk nomor urut
                counter = 1

                for index, row in earthquakes_felt.iterrows():
                    magnitude = row['Magnitude']
                    date = row['Tanggal'].strftime('%d %B %Y')  # Format tanggal DD/MM/YYYY
                    time = row['Waktu (WIB)']  # Asumsikan kolom waktu sudah dalam format string yang benar
                    location = row['Dirasakan']
                    description = row['Keterangan']
                    depth = row['Kedalaman']

                    # Format detail gempa yang dirasakan dengan nomor urut
                    felt_detail = (
                        f"{counter}. Gempabumi dengan magnitudo {magnitude} {location} yang terjadi pada tanggal {date} "
                        f"pukul {time} WIB berlokasi di {description} pada kedalaman {depth} km."
                    )

                    # Menambahkan detail gempa ke dalam dokumen sebagai paragraf terpisah
                    felt_detail_paragraph = document.add_paragraph(felt_detail)
                    felt_detail_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

                    # Meningkatkan counter untuk nomor urut berikutnya
                    counter += 1
            else:
                # Jika tidak ada gempa yang dirasakan, tambahkan keterangan tambahan
                no_felt_detail = document.add_paragraph("Tidak ada rincian gempabumi yang dirasakan selama periode ini.")
                no_felt_detail.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

            # Add space
            document.add_paragraph('')
            
            document.add_paragraph("Halo sobat BMKG Denpasar! Apa kabar? Semoga sehat ya🤗\n" 
                                f"Berikut kami informasikan aktivitas kegempaan wilayah Bali dan sekitarnya periode {nama_folder}."
                                "Pada minggu ini, aktivitas kegempaan didominasi oleh gempa magnitudo M < 3 dengan kedalaman dangkal < 60 km."
                                f"Pada minggu ini terdapat {total_earthquakes} kejadian gempabumi." \
                                "\n" \
                                "\n" \
                                f"Jumat, {end_date_formatted}.\n"
                                ".\n"
                                ".\n"
                                ".\n"
                                "Hello BMKG Denpasar friends! How are you? Hope you are healthy🤗\n" 
                                f"Here we provide information on earthquake activity in the Bali area and its surroundings for the period {start_date_formatted} – {end_date_formatted}."
                                "This week, seismic activity was dominated by earthquakes with a magnitude of M < 3 with a shallow depth of < 60 km."
                                f"This week there are {total_earthquakes} earthquake event. Greetings of health and enthusiasm." \
                                "\n" \
                                "\n" \
                                f"Friday, {end_date_formatted}.\n"
                                "#stageofdenpasar #dewata #dedikasi #wibawa #tanggap #akuntabel #siapuntukselamat #kukul #kentongan #linuh")

            # Add space
            document.add_paragraph('')
            
            document.add_paragraph("Halo sobat BMKG Denpasar! Apa kabar? Semoga sehat ya🤗.\n"
                                f"Berikut kami informasikan aktivitas kegempaan wilayah Bali dan sekitarnya periode {nama_folder}.\n"
                                "Salam sehat dan semangat. \n"
                                "#stageofdenpasar #dewata #dedikasi #wibawa #tanggap #akuntabel #siapuntukselamat ")


            # Add space
            document.add_paragraph('')

            # Tambah penerima
            document.add_paragraph("Yth.\n1. Ibu Deputi Bidang Geofisika\n"
                                "2. Bapak Direktur Gempabumi dan Tsunami\n"
                                "3. Bapak Direktur Seismologi Teknik, Geofisika Potensial dan Tanda Waktu\n"
                                "4. Bapak Kepala Balai Wilayah III\n")

            # Tambah salam pembuka
            document.add_paragraph("Dengan hormat,\nBerikut kami sampaikan sebagai berikut:")

            # Tambah poin pertama
            document.add_paragraph(f"1. Infografis, videografis dan data excel seismisitas wilayah Bali periode {nama_folder}. "
                                "Laporan lengkap kami kirimkan ke email: zona.potensi@gmail.com")

            # Tambah daftar link media sosial
            document.add_paragraph("\nBerikut terlampir link media sosial infografis dan videografis seismisitas wilayah Bali dan sekitarnya dari Stasiun Geofisika Denpasar:")

            social_links = [
                ("1. Link Instagram", "Videografis dan Infografis: https://www.instagram.com/bmkg_denpasar"),
                ("2. Link Facebook", "Videografis dan Infografis: https://web.facebook.com/BMKGDenpasar"),
                ("3. Link X", "Videografis dan Infografis: https://x.com/BMKG_Denpasar"),
            ]

            for title, link in social_links:
                p = document.add_paragraph()
                p.add_run(title).bold = True
                p.add_run("\n" + link)

            # Tambah penutup
            document.add_paragraph("\nDemikian, mohon berkenan dan terima kasih.")
            document.add_paragraph("Hormat kami,\nBMKG Stasiun Geofisika Denpasar")

            # Tambah kontak
            document.add_paragraph("\nKontak Stasiun Geofisika Denpasar:")
            contacts = [
                "📍 Alamat: Jl. Pulau Tarakan No. 1 Sanglah, Denpasar-Bali",
                "📞 Telepon: (0361) 226157",
                "🌐 Web: stageof-bali.bmkg.go.id",
                "📧 Email: stageof.denpasar@bmkg.go.id"
            ]

            for contact in contacts:
                document.add_paragraph(contact)

            # Tambah kalimat penutup
            document.add_paragraph("\nDemikian yang dapat kami sampaikan.\n\nTerima kasih.")

        # Simpan 'datalengkap.csv' ke folder baru dengan nama 'data gempa (nama folder).xlsx'
            output_file = os.path.join(nama_folder, f'Deskripsi info seismisitas website {nama_folder}.docx')
        
            # Save the document
            document.save(output_file)
            
        messagebox.showinfo("Cetak Deskripsi", "Berhasil Mencetak deskripsi")

    def force_exit(self):
        if messagebox.askokcancel("Keluar", "Yakin ingin keluar dari aplikasi?"):
            self.root.destroy()
            sys.exit()

if __name__ == "__main__":
    root = tk.Tk()
    app = EarthquakeApp(root)
    root.mainloop()


<>:711: SyntaxWarning: invalid escape sequence '\s'
<>:711: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Hype G12\AppData\Local\Temp\ipykernel_9424\1369658525.py:711: SyntaxWarning: invalid escape sequence '\s'
  cross = pd.read_csv('crsx.dat', delimiter='\s+', header=None)
Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Python312\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\Hype G12\AppData\Local\Temp\ipykernel_9424\1369658525.py", line 187, in process_data
    df[column_name] = df[column_name].astype(str).apply(self.standardize_months)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Python312\Lib\site-packages\pandas\core\series.py", line 4924, in apply
    ).apply()
      ^^^^^^^
  File "c:\Python312\Lib\site-packages\pandas\core\apply.py", line 1427, in apply
    return self.apply_standard()
           ^^^^^^^^^^^^^^^^^^^^^
  F